# Generation of Figures 1, 2, 3, 5 and 6 of the document entitled 
# **Rebuttal to “Comment on ‘A theoretical upper limit for offshore wind energy extraction’ by Simão Ferreira et al. (2026)”: reproducibility analysis, audit of the vdLW implementation, and clarification of finite wind-farm correction methodologies**

This script creates Figures 1, 2, 3, 5 and 6 of the document entitled 

**Rebuttal to “Comment on ‘A theoretical upper limit for offshore wind energy extraction’ by Simão Ferreira et al. (2026)”:** <br> 
**reproducibility analysis, audit of the vdLW implementation, and clarification of finite wind-farm correction methodologies**

by 

Jens Nørkær Sørensen, Gunner Chr. Larsen, and Carlos Simão Ferreira

first released on 2026 May 16th 

at https://wes.copernicus.org/preprints/wes-2026-59/  CC1: 'Comment on wes-2026-59', Jens Nørkær Sørensen, 16 May 2026

# Generation of Figures 1, 2 and 3 - Correction of Type 1 errors

This script creates Figures 1, 2 and 3 of the document entitled 

**Rebuttal to “Comment on ‘A theoretical upper limit for offshore wind energy extraction’ by Simão Ferreira et al. (2026)”:** <br> 
**reproducibility analysis, audit of the vdLW implementation, and clarification of finite wind-farm correction methodologies**

by 

Jens Nørkær Sørensen, Gunner Chr. Larsen, and Carlos Simão Ferreira

first released on 2026 May 16th 

at https://wes.copernicus.org/preprints/wes-2026-59/  CC1: 'Comment on wes-2026-59', Jens Nørkær Sørensen, 16 May 2026

## Import of necessary libraries

In [ ]:
# first, we will import the necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Definition and modification of the function by van der Paul and Watson

The original function was published by van der Laan and Watson (vdLW) in https://doi.org/10.5281/zenodo.19235300 , with the code in the file "run.py"

We could just import the function from the "run.py" file by vdLW. 
However, the function has an error in one of the equations. 
We need to modifiy the function to run with and without the error. 

Therefore, we copy here the function by vdLW and make clear what is modified. We just added a bolean flag to run the original eps2 formulation by vdLW and the correct one, added now by SLS. This is to correct Type 1 error 4 as described in the Rebuttal document.

In [ ]:
from scipy.special import gamma, gammainc 
from scipy.optimize import fsolve

class MinimalisticPredictionModel():
    """Sørensen, J.N.; Larsen, G.C.
    A Minimalistic Prediction Model to Determine Energy Production and Costs of Offshore Wind Farms.
    Energies 2021, 14, 448. https://doi.org/10.3390/en14020448"""

    def __init__(self, correction_factor, latitude, CP, Uin, Uout, rho, correct_eps2=False):
        ########################### Start of note by SLS ###########################
        # The original code by vdLW had the function header as: def __init__(self, correction_factor, latitude, CP, Uin, Uout, rho):
        # 
        # We have added an optional argument 'correct_eps2' to the function header. This argument is a boolean.
        # If False (default), the function will use the original calculation for eps2 as given by vdLW.
        # If True, the function will use the correct formula for eps2
        #
        ########################### End of note by SLS   ###########################
        """
        Parameters
        ----------
        correction_factor : int, float or function
            Finite-size wind farm corrrection which multiplied with sqrt(Nturb) gives
            the number of wind turbines exposed to the free wind
        latitude : int or float
            latitude [deg] used to calculate the coriolis parameter
        CP : float, optional
            Wind turbine power coefficient
        Uin : int or float, optional
            Wind turbine cut-in wind speed
        Uout : int or float, optional
            Wind turbine cut-out wind speed
        rho : float, optional
            Air density
        """

        self.CP = CP
        self.Uin = Uin
        self.Uout = Uout
        omega = 2 * np.pi / (24 * 60 * 60)  # earth rotation speed
        self.f = 2 * omega * np.sin(np.deg2rad(latitude))
        self.correction_factor = correction_factor
        self.rho = rho

        ################ Start of code alteration by SLS to add correct_eps2 argument, see note at the beginning of the class #################
        self.correct_eps2 = correct_eps2
        ################ End of code alteration by SLS to add correct_eps2 argument #################
        
    def predict(self, Pg, CT, D, H, z0, Aw, kw, Nturb, Area):
        """
        Inputs:
            Pg    - [W] Nameplate capacity (generator power)
            CT    - [-] Thrust coefficient
            D     - [m] Rotor diameter
            H     - [m] Tower height
            z0    - [m] roughness length
            Aw    - [m/s] Weibull scale parameter
            kw    - [-] Weibull shape parameter
            Nturb - [-] Number of turbines
            Area  - [m2] Area of wind farm

        Outputs:
            power - [Wh] Annual energy production of the wind farm
            ws_eff - [m/s] Effective mean wind speed including wakes
        """

        kappa = 0.4  # [-] Von Karman constant
        Uin, Uout = self.Uin, self.Uout

        # factor defined by Frandsen, should be used instead of f in eq 13 and 19 (typos in paper)
        fm = self.f * np.exp(4)
        delta = np.log(H / z0)  # eq 19

        # Mean spacing between wt in diameters, eq 8
        S = np.sqrt(Area) / (D * (np.sqrt(Nturb) - 1))

        # Rated wind speed [m/s], eq 4
        Ur = (8 * Pg / (self.rho * np.pi * D**2 * self.CP))**(1 / 3)

        # Power modeled as P = alpha * U^3 + beta, eq 1
        alpha = Pg / (Ur**3 - Uin**3)  # [(m/s)^-3] eq 2
        beta = -Pg * Uin**3 / (Ur**3 - Uin**3)  # [-], eq 2

        Uh0 = Aw * gamma(1 + 1 / kw)  # [m/s] Mean velocity at hub height
        Ctau = np.pi * CT / (8 * S * S)  # [-] Wake parameter, rotor ct smeared on WT area
        nu = np.sqrt(0.5 * Ctau) * D / (kappa**2 * H) * delta  # [-] wake eddy viscosity

        # Finite-size wind farm corrrection, section 2.5
        correction_factor = self.correction_factor
        if hasattr(correction_factor, '__call__'):
            correction_factor = correction_factor(Uh0, S, Nturb)
        Nfree = correction_factor * np.sqrt(Nturb)  # Number of wt exposed to the free wind
        Nfree = np.minimum(Nfree, Nturb)  # To make sure Nfree/Nturb is not larger than one, not yet implemented in PyWake

        # Geostrophic wind speed
        G_last = Uh0
        for n in range(10):
            G = Uh0 * (1 + np.log(G_last / (fm * H)) / delta)
            dG = abs(G - G_last)
            if dG < 1e-5:
                break
            G_last = G

        gam = np.log(G / (fm * H))  # eq 19

        # Mean velocity at hub height without wake effects from geostrophic wind
        Uh0 = G / (1 + gam / kappa * np.sqrt((kappa / delta)**2))  # eq 13, ct=0

        # Power without wake effects, eq 16 modified by
        # - add gamma(1 + 3 / kw) to cancel out normalization in scipy's gammainc
        # - gammainc terms swapped (typo in paper) 
        def get_Py(Aw, Aw_out):  # Yearly power
            return alpha * Aw**3 * gamma(1 + 3 / kw) * (gammainc(1 + 3 / kw, (Ur / Aw)**kw) - gammainc(1 + 3 / kw, (Uin / Aw)**kw)) +\
                beta * (np.exp(-(Uin / Aw)**kw) - np.exp(-(Ur / Aw)**kw)) + \
                Pg * (np.exp(-(Ur / Aw)**kw) - np.exp(-(Uout / Aw_out)**kw))

        P_y = get_Py(Aw, Aw)
 
        # Without cutin and cutout
        # def get_Py(Aw):  # Yearly power
        #     x = (Ur / Aw) ** kw
        #     ks = 1 + 3 / kw
        #     return Pg * (x ** (1.0 - ks) * gamma(ks) * gammainc(ks, x) + np.exp(-x))
        # def get_Py(Aw):  # Yearly power
        #     x = (Ur / Aw)
        #     ks = 1 + 3 / kw
        #     return Pg * (x ** (-3) * gamma(ks) * gammainc(ks, x ** kw) + np.exp(-x ** kw))                      
        # P_y = get_Py(Aw)

        # Mean velocity at hub height with wake effects
        z0_lo = z0  # / (1 - D / (2 * H))**(nu / (1 + nu))  # ???
        Uh = G / (1 + gam * np.sqrt(Ctau + (kappa / np.log(H / z0_lo))**2) / kappa)

        # eq 18. The paper states 3/2 instead of 3.2 which is either a typo or an initial guess
        # eps2 corresponds to eps(Uout) in paper and eps2(Ur)=eps1
        eps1 = (1 + gam / delta) / (1 + gam / kappa * np.sqrt(Ctau + (kappa / delta)**2))
        eps2 = (1 + gam / delta) / (1 + gam / kappa * np.sqrt(Ctau * (Ur / Uh)**3.2 + (kappa / delta)**2))


        ################# Start of code alteration by SLS to correct eps2, see note at the beginning of the class #################
        if self.correct_eps2: # 
            # use the correct formulation
            eps2 = (1 + gam / delta) / (1 + gam / kappa * np.sqrt(Ctau * (Ur / Uout)**3.2 + (kappa / delta)**2))
        ################### End of code alteration by SLS to correct eps2 #################




        # print('e', eps1, correction_factor)
        # Power production with wake effects
        P_WFy = get_Py(eps1 * Aw, eps2 * Aw)

        power = ((Nturb - Nfree) * P_WFy + Nfree * P_y)
        ws_eff = ((Nturb - Nfree) * Uh + Nfree * Uh0) / Nturb
        
        # Phi without cutin and cutout 
        def get_phi(x):  # Yearly power
            # x = U_r / (Uh0 eps)
            ks = 1 + 3 / kw      
            Gm = gamma(1 + 1 / kw)
            return Pg * ((x * Gm) ** (-3) * gamma(ks) * gammainc(ks, (x * Gm) ** kw) + np.exp(-(x * Gm) ** kw)) - power / Nturb
            
        root = fsolve(get_phi, x0=1)  # initial guess        
        phi = root[0]

        return power, ws_eff, P_y, P_WFy, Uh, G, phi


## Definition of a function to run all wind farm cases, and return the capacity factors

This function receives all input data for each wind farm case, and loops through all the cases. For each case, it creates an instance of the class **MinimalisticPredictionModel** by vdLW, and calculates the different capacity factors (isolated widn turbine, infinite wind farm and finite wind farm)

In [ ]:
def run_code_all_cases(Turbine_rated_power, CT, Turbine_rotor_diameter, Turbine_hub_height, z0, Wind_lambda, kw, Windfarm_N_turbines, Windfarm_area,
                       correction_factor, latitude, CP, Uin, Uout, rho, correct_eps2=False):
    
    # function created by SLS to run the code for all cases, this is done to avoid having to run the code for each case separately in the notebook, 
    # which would make it more difficult to read and understand the code. 
    # This function takes as input the parameters for all cases and returns the outputs for all cases in arrays.


    # define the number of cases that the code will be run for, this is done by taking the length of the Windfarm_area array, 
    # but it can be any of the wind farm related input arrays as they should all have the same length
    n_cases = len(Windfarm_area)

    # crate arrays for the outputs, now zeros with the same length as the number of cases 
    # first, arrays for the outputs of the predict function, these will be filled in the loop below, and then used to calculate the correction factors
    power = np.zeros(n_cases) 
    ws_eff = np.zeros(n_cases)
    P_y = np.zeros(n_cases)
    P_WFy = np.zeros(n_cases)
    Uh = np.zeros(n_cases)
    G = np.zeros(n_cases)
    phi = np.zeros(n_cases)

    # now, arrays to return as output of this function, these will be filled in the loop below using the outputs of the predict function
    Cf_Free = np.zeros(n_cases)
    Cf_Inf = np.zeros(n_cases)
    Cf_Model = np.zeros(n_cases)


    # we loop over the number of cases, and for each case we run the code and fill the output arrays
    for i in range(n_cases):
        # check if the input is a single value or an array, if an arry use the i-th value, if a single value use it directly
        # this is done to allow for both cases where the input is a single value (same for all cases) or an array (different for each case)
        correction_factor_i = correction_factor[i] if isinstance(correction_factor, (list, np.ndarray)) else correction_factor
        latitude_i = latitude[i] if isinstance(latitude, (list, np.ndarray)) else latitude
        CP_i = CP[i] if isinstance(CP, (list, np.ndarray)) else CP
        Uin_i = Uin[i] if isinstance(Uin, (list, np.ndarray)) else Uin
        Uout_i = Uout[i] if isinstance(Uout, (list, np.ndarray)) else Uout
        rho_i = rho[i] if isinstance(rho, (list, np.ndarray)) else rho
        Turbine_rated_power_i = Turbine_rated_power[i] if isinstance(Turbine_rated_power, (list, np.ndarray)) else Turbine_rated_power
        CT_i = CT[i] if isinstance(CT, (list, np.ndarray)) else CT
        Turbine_rotor_diameter_i = Turbine_rotor_diameter[i] if isinstance(Turbine_rotor_diameter, (list, np.ndarray)) else Turbine_rotor_diameter
        Turbine_hub_height_i = Turbine_hub_height[i] if isinstance(Turbine_hub_height, (list, np.ndarray)) else Turbine_hub_height
        z0_i = z0[i] if isinstance(z0, (list, np.ndarray)) else z0
        Wind_lambda_i = Wind_lambda[i] if isinstance(Wind_lambda, (list, np.ndarray)) else Wind_lambda
        kw_i = kw[i] if isinstance(kw, (list, np.ndarray)) else kw
        Windfarm_N_turbines_i = Windfarm_N_turbines[i] if isinstance(Windfarm_N_turbines, (list, np.ndarray)) else Windfarm_N_turbines
        Windfarm_area_i = Windfarm_area[i] if isinstance(Windfarm_area, (list, np.ndarray)) else Windfarm_area

        # create an instance of the MinimalisticPredictionModel class with the input parameters for this case, 
        # and then call the predict function to get the outputs for this case, and fill the output arrays
        predictionmodel_vdLW = MinimalisticPredictionModel(correction_factor_i, latitude_i, CP_i, Uin_i, Uout_i, rho_i, correct_eps2=correct_eps2)

        power[i], ws_eff[i], P_y[i], P_WFy[i], Uh[i], G[i], phi[i] =  predictionmodel_vdLW.predict(Turbine_rated_power_i, CT_i, Turbine_rotor_diameter_i, Turbine_hub_height_i, z0_i, Wind_lambda_i, kw_i, Windfarm_N_turbines_i, Windfarm_area_i)

        Cf_Free[i] = P_y[i] / Turbine_rated_power_i
        Cf_Inf[i] = P_WFy[i] / Turbine_rated_power_i
        Cf_Model[i] = power[i] / (Turbine_rated_power_i * Windfarm_N_turbines_i)


    return Cf_Free, Cf_Inf, Cf_Model












## Read input data from Output/Input files by vdLW

vdLW, with their code, create a file named "Output.csv", which contains both inputs and outputs used by vdLW, including the inputs used by SLS.

An important note is that vdLW do not use all the wind farm cases. vdLW choose not to use the cases of the wind farms "Baltic 1", "Baltic 2", "Princess Amalia", "Luchterduinen". Therefore, after reading the files, we will need to remove those cases.

In [ ]:
# read output/input data from vdLW's code
df = pd.read_csv("output.csv")
db_first_eval = df.copy() # this is just not to destroy the original dataframe, as we will be modifying it in the next steps

# we will eliminate the cases of Baltic I and II and Princess Amalia and Lucherduinen, as they are not evaluated by vdLW's work
db_first_eval = db_first_eval[~db_first_eval["Name"].isin(["Baltic 1", "Baltic 2", "Princess Amalia", "Luchterduinen"])]

# display the entire database for the first evaluation, indicating explicitly to plot all the rows, to avoid the Jupyter notebook from truncating the output
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
display(db_first_eval)

In [ ]:
# we can now read the input parameters for all wind farm cases
latitude = db_first_eval["lat"].values
Windfarm_area = db_first_eval["A"].values*1e6
Windfarm_N_turbines = db_first_eval["Nt"].values
Windfarm_rated_power = db_first_eval["MW install"].values*1e6
Turbine_hub_height = db_first_eval["Ht"].values
Turbine_rotor_diameter = db_first_eval["D"].values
Turbine_rated_power = db_first_eval["PG"].values*1e6
Wind_lambda = db_first_eval["lambda"].values
Wind_k= db_first_eval["k"].values
Wind_reference_height = db_first_eval["Href"].values

# correction factor for the Nfree turbines
NFree = np.array(db_first_eval['Nrowscale parameter'] * db_first_eval['Nturb_frontal_area'], dtype=float)
correction_factor = NFree / np.sqrt(Windfarm_N_turbines)   


## Here, we define the fixed input parameters, as defined by vdLW and the ones that need correction  


In [ ]:
# parameters as defined by vdLW, these are the same for all cases, as they are not case specific, 
# but they are used as input for the predict function, so we define them here
vdLW_z0 = 10 ** (-4)  # Roughness length [m]
vdLW_zRef = 100.0  # Reference height at which GWA data is taken [m]
vdLW_kw = 2.4  # Weibull shape parameter [-]
vdLW_rho = 1.225  # Air density [kg/m^3]
vdLW_lat = 55  # Latitude [degree]
vdLW_CP = 0.46  # Turbine power coefficient [-]
vdLW_CT = 0.75  # Turbine thrust coefficient [-]
vdLW_Uin = 0.0  # Cut-in wind speed [m/s]
vdLW_Uout = 1e6  # Cut-out wind speed [m/s]

# correct inputs by SLS, as used in the original paper by SLS
SLS_Uin = 3.0  # Cut-in wind speed [m/s]
SLS_Uout = 25.0  # Cut-out wind speed [m/s]
SLS_lat = latitude  # Latitude [degrees] , it is wind farm specific 


## Correcting Type 1 error 1 : removal by vdLW of the hub-height wind-speed correction of the Weibull scale parameter, causing a different input of the local unperturbed wind speed

The reference $\lambda$ (scale factor of the Weibull distribution) is read at a refernce height. It need to be corrected to the hub height. vdLW originally also did this, but then it was "commneted out" of their code. This is a very important error. To correct it, its a single line of code

In [ ]:
# correction of Type 1 error 1: removal by vdLW of the hub-height wind-speed correction of the Weibull scale parameter, causing a different input of the local unperturbed wind speed

Wind_lambda_corrected_to_hub_height = Wind_lambda * np.log(Turbine_hub_height / vdLW_z0) / np.log(Wind_reference_height / vdLW_z0)

## Calculation of all wind farm cases: original vdLW formulation, and then with the errors corrected

### calculation of the original vdLW predictions for all cases

In [ ]:
# calculation of the original vdLW predictions for all cases, using the original inputs as given by vdLW, and using Nfree as determined bu SLS
Cf_Isolated_vdLW, Cf_InfWindFarm_vdLW, Cf_Model_vdLW =run_code_all_cases(Turbine_rated_power, vdLW_CT, Turbine_rotor_diameter, Turbine_hub_height, vdLW_z0, Wind_lambda, vdLW_kw, Windfarm_N_turbines, Windfarm_area,
                       correction_factor, vdLW_lat , vdLW_CP, vdLW_Uin, vdLW_Uout, vdLW_rho, correct_eps2=False)

### Check that this generates the results published by vdLW, to be sure that all is correct

First, we will read from the dataframe of "Output.cvs" (the output file generated by vdLW) the results by vdLW, and then we compare with the results published above 


In [ ]:
# results calculated and published by vdLW
vdLW_Cf_model_published = np.array(db_first_eval["Cf_Model"].values, dtype=float)


# we will plot the results of the original vdLW predictions against the published results by vdLW, to check if we can reproduce their results, and to see if there are any discrepancies
plt.figure(figsize=(10, 6))

# the Cf_model is multiplied by the ratio of the total rated power of the wind farm to the rated power of the turbine, to get the same units as the published results by vdLW, 
# for 6 wind farms, there is a difference between the value of nameplate capacity of the wind farm and the product of rated wind turbine power and number of turbines (rounding and typos)
# in the code of vdLW, they use the nameplate capacity of rated power of the wind farm, not N turbines*Turbine rated power
# the correct approach is to always use the number of turbines and the rated power of each turbine, 
# as in a few times, the announced nameplate capacity of the wind farm is larger than the product of number of turbines and rated power of each turbine
# for the purpose of checking the code, this has no impact.

plt.plot(Cf_Model_vdLW*(Windfarm_N_turbines*Turbine_rated_power)/Windfarm_rated_power, vdLW_Cf_model_published, 'o', label='vdLW current code vs vdLW published')
# plot a 1:1 line for reference
plt.plot([0, 1], [0, 1], 'k--', label='1:1 line')
plt.xlabel('Cf Model by vdLW,  current code in this notebook')
plt.ylabel('Cf Model Published by vdLW')
plt.xlim(0.3, 0.7)
plt.ylim(0.3, 0.7)
plt.grid(True)
plt.legend()
plt.show()


### calculations with the Type 1 errors corrected

In [ ]:
### calculations with the Type 1 errors corrected


# correct TYpe 1 error 1: correction of the hub-height wind-speed correction of the Weibull scale parameter, 
# causing a different input of the local unperturbed wind speed,     
Cf_Free_vdLW_correct_1, Cf_Inf_vdLW_correct_1, Cf_Model_vdLW_correct_1 =run_code_all_cases(Turbine_rated_power, 
                vdLW_CT, Turbine_rotor_diameter, Turbine_hub_height, vdLW_z0, Wind_lambda_corrected_to_hub_height, 
                vdLW_kw, Windfarm_N_turbines, Windfarm_area,
                correction_factor, vdLW_lat , vdLW_CP, vdLW_Uin, vdLW_Uout, vdLW_rho, correct_eps2=False)


#correct Type 1 error 1 and 2: correction of the cut-in and cut-out wind speeds, using the correct cut-in and cut-out wind speeds as defined by SLS, 
Cf_Free_vdLW_correct_1_and_2, Cf_Inf_vdLW_correct_1_and_2, Cf_Model_vdLW_correct_1_and_2 =run_code_all_cases(Turbine_rated_power, 
                    vdLW_CT, Turbine_rotor_diameter, Turbine_hub_height, vdLW_z0, Wind_lambda_corrected_to_hub_height, vdLW_kw, 
                    Windfarm_N_turbines, Windfarm_area,
                    correction_factor, vdLW_lat , vdLW_CP, SLS_Uin, SLS_Uout, vdLW_rho, correct_eps2=False)

#correct Type 1 errors 1,2 and 3: using correct latitude for each wind farm case
Cf_Free_vdLW_correct_1_and_2_and_3, Cf_Inf_vdLW_correct_1_and_2_and_3, Cf_Model_vdLW_correct_1_and_2_and_3 =run_code_all_cases(
    Turbine_rated_power, vdLW_CT, Turbine_rotor_diameter, Turbine_hub_height, vdLW_z0, Wind_lambda_corrected_to_hub_height, 
    vdLW_kw, Windfarm_N_turbines, Windfarm_area,
    correction_factor, SLS_lat , vdLW_CP, SLS_Uin, SLS_Uout, vdLW_rho, correct_eps2=False)


# correct TYpe 1 errors 1,2,3 and 4: correction of the eps2 formulation, using the correct formulation for eps2 as defined by SLS, which is based on the cut-out wind speed instead of the rated wind speed, as used by vdLW
Cf_Free_vdLW_correct_all, Cf_Inf_vdLW_correct_all, Cf_Model_vdLW_correct_all =run_code_all_cases(Turbine_rated_power, 
                    vdLW_CT, Turbine_rotor_diameter, Turbine_hub_height, vdLW_z0, Wind_lambda_corrected_to_hub_height, 
                    vdLW_kw, Windfarm_N_turbines, Windfarm_area,
                    correction_factor, SLS_lat, vdLW_CP, SLS_Uin, SLS_Uout, vdLW_rho, correct_eps2=True)

# corrected formulation, by limit case with Uin=0 and Uout=infinity
Cf_Free_vdLW_correct_Uin0_Uoutinf, Cf_Inf_vdLW_correct_Uin0_Uoutinf, Cf_Model_vdLW_correct_Uin0_Uoutinf =run_code_all_cases(Turbine_rated_power, 
                    vdLW_CT, Turbine_rotor_diameter, Turbine_hub_height, vdLW_z0, Wind_lambda_corrected_to_hub_height, 
                    vdLW_kw, Windfarm_N_turbines, Windfarm_area,
                    correction_factor, SLS_lat, vdLW_CP, 0, 10e6, vdLW_rho, correct_eps2=True)




## Read the results published by SLS 2026, from comparison and generation of the figures

We already removed the wind farms "Baltic 1", "Baltic 2", "Princess Amalia", "Luchterduinen" because vdLW removed them from their work

In [ ]:
# read from excel file the dataframe of SLS_2026
df_SLS_2026 = pd.read_excel("SLS_2026_without_Baltic1_2_LuchD_PAM.xlsx", sheet_name="Sheet1")

# createa array of capacity factors from the dataframe of SLS_2026, 
# this will be used to compare with the results of the corrected vdLW model
SLS_2026_CF = np.array(df_SLS_2026["Capacity Factor model"].values, dtype=float)/100


## Plot figure 1

In [ ]:
# Plot figure 1
plt.rcParams.update({'font.size': 14}) # fontsize 14 for all labels and ticks

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(SLS_2026_CF*100, Cf_Model_vdLW*100, 'o', color='red', markersize=4, label=r'vdLW  original')

ax.plot(SLS_2026_CF*100, np.round(Cf_Model_vdLW_correct_all*100, 1), '*', color='green', markersize=10,
        label=r'vdLW, with the four errors corrected')
ax.plot([0., 100.], [0., 100.], "k--", alpha=0.5, label="1:1 line")
ax.set_xlim(30, 60)
ax.set_ylim(30, 60)
ax.set_xlabel('Capacity Factor published by  SLS (%)')   
ax.set_ylabel('Capacity Factor by vdLW code (%)')
ax.legend()
plt.show()

## Plot figure 2

In [ ]:
# same, but with 2025 data
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(SLS_2026_CF, (Cf_Model_vdLW-SLS_2026_CF)/SLS_2026_CF*100, 'o', color='black', markersize=10, label=r'vdLW  original')
ax.plot(SLS_2026_CF, (Cf_Model_vdLW_correct_1-SLS_2026_CF)/SLS_2026_CF*100, 'o', color='red', markersize=8,
        label=r'vdLW, with corrected Error 1 — Removal of the Hub-Height Wind-Speed Correction')
ax.plot(SLS_2026_CF, (Cf_Model_vdLW_correct_1_and_2-SLS_2026_CF)/SLS_2026_CF*100, 'o', color='orange', markersize=6, 
        label=r'vdLW, with corrected Error 1 and Error 2 — Incorrect Use of Cut-in and Cut-out Wind Speeds')
ax.plot(SLS_2026_CF, (Cf_Model_vdLW_correct_1_and_2_and_3-SLS_2026_CF)/SLS_2026_CF*100, 'o', color='yellow', markersize=4,
        label=r'vdLW, with corrected Errors 1, 2 and Error 3 — Use of a Fixed Latitude Instead of Site-Specific Latitude')
ax.plot(SLS_2026_CF, (np.round(Cf_Model_vdLW_correct_all, 3)-SLS_2026_CF)/SLS_2026_CF*100, '*', color='green', markersize=10,
        label=r'vdLW, all errors corrected (Error 4 — Incorrect Equation in the Capacity-Factor Calculation)')
ax.plot([0., 1], [0., 0.], "k--")
ax.set_xlim(0.3, 0.6)
ax.set_ylim(-2, 15)
ax.legend(fontsize=12)
ax.set_xlabel('Capacity Factor published by  SLS (%)')    
# ax.set_title('Relative difference between Capacity Factor of Paul\'s model and repository 2025 data \n  (in % of capacity factor 2025)')
ax.set_ylabel('Relative difference between vdLW code and SLS data \n (as % of capacity factor SLS data) ')
ax.grid()
# ax.legend()
plt.show()


## Create Figure 3

First, we need to read the Nfree correction by vdLW, and then apply it to the correct capacity factors, to obtain the result without Type 1 errors

In [ ]:
# get Nedge turbines, calculated by vdLW
Nedge_Neighbours = np.array(db_first_eval['Nwt_row_wf_neighbors'].values, dtype=float)
# vdLW assume a factor of 2.5 to determine the fraction of power of isolated turbines
m_afactor_vdLW = 2.5
a_factor_vdLW_Nfree=m_afactor_vdLW*Nedge_Neighbours/(Windfarm_N_turbines)


# calculate the fraction of power of isolated turbines, as determined by vdLW, using the Nedge_Neighbours and the factor of 2.5
CF_vdLW_Type1_corrected_NfreevdLW = a_factor_vdLW_Nfree*Cf_Free_vdLW_correct_all + (1-a_factor_vdLW_Nfree)*Cf_Inf_vdLW_correct_all



In [ ]:
# simple code for the linear fit

def linear_fit(x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    # fit y = a*x + b
    a, b = np.polyfit(x, y, 1)

    # predictions
    y_pred = a * x + b

    # R^2
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r2 = 1 - ss_res / ss_tot

    return a, b, r2

Now, we plot the figure

In [ ]:
# determine terms of linear regression between the capacity factor of vdLW with Type 1 error corrected Nfree and the SLS_2026_CF, 
# to see how well they correlate, and to see if there is a bias in the results
a_reg,b_reg,r2_reg = linear_fit(SLS_2026_CF, CF_vdLW_Type1_corrected_NfreevdLW)
print("Linear fit: a =", a_reg, "b =", b_reg, "R^2 =", r2_reg)



# we will now plot in the comparison between model and real CF
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(SLS_2026_CF*100,CF_vdLW_Type1_corrected_NfreevdLW*100, 'o',color='orange',  
        label=f'vdLW Nfree methodology, Type 1 errors corrected. Linear regression fit: a={a_reg:.2f}, b={b_reg:.2f}, R^2={r2_reg:.2f}')
ax.plot([0., 100.], [0., 100.], "k--", label="1:1 line")

ax.set_xlim(20, 70)
ax.set_ylim(20, 70)
ax.set_xlabel('Capacity Factor SLS (%)')
ax.set_ylabel('Capacity Factor vdLW (%)')
ax.legend()
# ax.set_title('Comparison of modelled and 2025 and Paul')
ax.grid()
plt.show()

## Export calculated data 
We will create a dataframe with correct capacity factors to use in the next analysis, and save it

In [ ]:
# we will create a dataframe, with the name of each case, and then with the corrected capacity factors Cf_Free_vdLW_correct_all, 
# Cf_Inf_vdLW_correct_all, Cf_Model_vdLW_correct_all, and also the corrected values for the limit case Uin=0 and Uout=infinity
df_corrected_vdLW = pd.DataFrame({
    "Name": db_first_eval["Name"].values,
    "Cf_Free_vdLW_correct_all": Cf_Free_vdLW_correct_all,
    "Cf_Inf_vdLW_correct_all": Cf_Inf_vdLW_correct_all,
    "Cf_Model_vdLW_correct_all": Cf_Model_vdLW_correct_all,
    "Cf_Model_vdLW_correct_Uin0_Uoutinf": Cf_Model_vdLW_correct_Uin0_Uoutinf,
    "Cf_Free_vdLW_correct_Uin0_Uoutinf": Cf_Free_vdLW_correct_Uin0_Uoutinf, 
    "Cf_Inf_vdLW_correct_Uin0_Uoutinf": Cf_Inf_vdLW_correct_Uin0_Uoutinf
})

# save to an excel file
df_corrected_vdLW.to_excel("vdLW_corrected_capacity_factors.xlsx", index=False)


##################################################################################################################################################

# Generation of Figures 5 and 6: correction of some Type 2 errors for some of the cases the work of vdLW

This script creates Figures 5 and 6 of the document entitled 

**Rebuttal to “Comment on ‘A theoretical upper limit for offshore wind energy extraction’ by Simão Ferreira et al. (2026)”:** <br> 
**reproducibility analysis, audit of the vdLW implementation, and clarification of finite wind-farm correction methodologies**

by 

Jens Nørkær Sørensen, Gunner Chr. Larsen, and Carlos Simão Ferreira

first released on 2026 May 16th 

at https://wes.copernicus.org/preprints/wes-2026-59/  CC1: 'Comment on wes-2026-59', Jens Nørkær Sørensen, 16 May 2026

## Import of necessary libraries

In [ ]:
# first, we will import the necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
from scipy.interpolate import interp1d

## Import of the code of vdLW as a library

We import the code of vdLW as a library to use the function.

However, there will be three functions by vdLW that we will copy into this notebook and modify:

a) The wind resource function, which prepares the input of the wind rose at the location of the wind farm. 
    This is for trwo reasons: 1) vdLW use the direction rose, while the orrect formulation is to use the wind power rose; and b) increase the directional intergration resolution, which is not only more accurate, it also helps mitigate some of the geometry errors in the vdLW code. The impact of these changes is very small for most cases, but its important for the case of complex cluesters in regions of a single predominant wind direction.

b) The function to calculate the number of Nedge turbines with neighbors and wind direction. The modification is only for data management, as the code by vdLW as many hardwired elements, and the modification allows us to calculate only the cases we are interested, instead of having to calcualte all the cases. This helps to keep this demonstration notebook cleaner. The change does not change the output of the original version not the method by vdLW.

c) The function to get coordinates from the inputfile, as it has an unnecessary variable hardwired in the code

In [ ]:
# here we import the code by vdLW in the run.py they published in zenodo, so we can use the function we need
import run as vdLWcode;

# the figure below is generated by the initiation of the vdLW code, it is not relevant

## Import wind farm and wind turbine data from database file published by vdLW

Here we upload the wind farm and wind turbine database used by vdLW in their code. As vdLW, we eliminate the same turbines vdLW eliminated.

The "data" folder is directly extracted from the Zenodo repository of the code by vdLW without modifications.


In [ ]:
# name of the input file, which is the csv file that contains the turbine data. We will read this file using pandas and store it in a dataframe called wtdata.
infile = 'data\\TurbineDatabase\\20251218_eww_opendatabase.csv'
# infile = 'data/TurbineDatabase/20251218_eww_opendb_carlos_corrected.xlsx'
wtdata = pd.read_csv(infile)

# duplicate the action by vdLW to remove these turbines
# Remove two turbines from Robin Rigg which have been decommissioned
# https://www.sciencedirect.com/science/article/pii/S0267726123000489
# 5026                   54.7715     -3.7006
# 5028                   54.7692     -3.7093    
wtdata = wtdata.drop([5026, 5028])  


## Import wind farm case list and data

We will use the same file that vdLW used, the only thing needed are the coordinated of the wind farm

In [ ]:
infile = 'data\\2026-02-11_input_vars_send_by_CarlosSimaoFerreira.csv'
inputdata = pd.read_csv(infile)   


## Import GWA wind resource data

This data was previously imported by vdLW and stored in the "data" folder. The code below are the function by vdLW to read the GWA data they stored. This is the procedure in their original code.

Below is the modified function, extracted from the code by vdLW. The changes are explained, and the original code is shown and commented

In [ ]:
# Original code by vdLW, to read the coordinates
# def get_latlon_decimal(coords):
#     nWF = len(coords)
#     latlon = np.zeros((nWF, 2)) 
#     for i in range(nWF):
#         latlon[i, 0] , latlon[i, 1]  = dms_string_to_decimal(inputdata['Coordinates'][i])   
#     return latlon

# issue: hardwired use of inputdata['Coordinates'] whcih are passed in as coords

def get_latlon_decimal(coords):
    nWF = len(coords)
    latlon = np.zeros((nWF, 2)) 
    for i in range(nWF):
        latlon[i, 0] , latlon[i, 1]  = vdLWcode.dms_string_to_decimal(coords[i])   
    return latlon

In [ ]:
# do not rerun the gwa wind resource, use data stored by vdLW
rerun_gwa = False

# Get list of lat lon in decimal format
latlon = get_latlon_decimal(inputdata['Coordinates'])
# Get Global Wind Atlas wind resource for each wind farm
# get_gwa_windresource(latlon, inputdata['Name'], rerun=True)    
ds = vdLWcode.get_gwa_windresource(latlon, inputdata['Name'], rerun=rerun_gwa)    

## Definition of the cases of wind farms for which we want to correct some Type 2 errors via preprocessing and inputs

There are 11 wind farms for which we can partially mitigate the Type 2 errors in the method and code by vdLW, without changing the code in the core vdLW method part. These are:

1. Gemini - Error: the wind farm is composed of two disjointed sub-wind farms. The method of vdLW creates an artificial wind farm space in the are inbetween, creating artificial. Solution: we will eliminate the administrative-induced error by labelling the subfarms as 1 and 2. 
2. Thortonbank - Same as Gemini
3. Borssele I-II - Error: missign wind farm Borssele III-V, which also affect the other farms in the Dutch-Belgium cluster. Solution: we will add the Borssele III-V as neighbours and target wind farms 
4. Norther - Same as above
5. Northwester 2 - Error: Neighbor geometries incorrect and wind power rose not applied. Solution: correct neighbor geometry and wind power rose applied. 
6. Belwind Phase 1 - Error: the method of vdLW takes the disjointed geoemtry of Nobelwind adn connects the two parts, creating an artificial wind farm area. This artifical area overlaps Belwind, creating an artificial infinite win-farm effect.
7. Nobelwind - Error: the issues above, but also an overestimation of the number of total free turbines in the West region of Nobelwind, by the use of 2.5*Nedge turbines in a single-row sub-windfarm. 
8. Northwind - Error: fails to identify the edge turbines due to the concave/convex limitations of the vdLW approach. Solution: use the solution already implemented by vdLW of adding a larger concativity allowance factor, which vdLW applied to other farms but not these ones (or used a low value).
9. Rentel - same as above  
10. Rampion - same as above
11. Robin Rigg - same as above 

8 out of 11 of these cases are in the Dutch-Belgium cluster. 


## Map of the Belgium-Dutch cluster

As 8 out of 11 of these cases are in the Dutch-Belgium cluster, it might help the reader to understand the completity of the geometry, and why the code by vdLW will have so much difficulties dealing with the geomwtries. We leave the representation below for support. Also, a representation of the local wind power rose.

![Figure 1](images_markdown/fig1.png)

![Figure 2](images_markdown/fig2.png)



## Representation of Gemini

One of the main issues in the vdLW code concerns the treatment of wind farms composed of non-contiguous sub-farms. The figure below shows a representation of the Gemini wind farm (and neighbors). The following figure was generated and published by vdLW in Zenodo, and shows how the creation of the artificial widn farm space and edge leads to underestimation of the number of wind turbines experiencing clean inflow (pink arrow represent wind direction). 

![Figure 3](images_markdown/fig3.png)

![Figure 4](images_markdown/fig4.png)


### Definition of alphas for the concave/convex edge geometry process by vdLW

We copy the original inputs defined by vdLW and add/modify for a few widn farms where the failure to identify the edge is highly relevant

In [ ]:
## Original code inputs by vdLW, we made into a function, so we can call it with different alphas and different administrative boundaries

    ## Original code by vdLW, to define the alphas for the concave hull method
    # # A Concave hull method is used to determine the farm edge, which is not unique and depends on alpha.
    # # A larger alpha means more concave. A too large alpha can result in strange results.
    # alphas = [0.0001] * len(inputdata['Nt'])
    # alphas[inputdata.loc[inputdata["Name"] == 'Amrumbank West', :].index[0]] = 0.0005        
    # alphas[inputdata.loc[inputdata["Name"] == 'Kriegers Flak', :].index[0]] = 0.00025
    # alphas[inputdata.loc[inputdata["Name"] == 'Beatrice extension', :].index[0]] = 0.0005    
    # alphas[inputdata.loc[inputdata["Name"] == 'Belwind Phase 1', :].index[0]] = 0.0005        
    # alphas[inputdata.loc[inputdata["Name"] == 'Borssele I-II', :].index[0]] = 0.0005       
    # alphas[inputdata.loc[inputdata["Name"] == 'DanTysk', :].index[0]] = 0.0005    
    # alphas[inputdata.loc[inputdata["Name"] == 'Dudgeon', :].index[0]] = 0.0005
    # alphas[inputdata.loc[inputdata["Name"] == 'Galloper', :].index[0]] = 0.00015  
    # alphas[inputdata.loc[inputdata["Name"] == 'Gode 1 and 2', :].index[0]] = 0.0005
    # alphas[inputdata.loc[inputdata["Name"] == 'Greater Gabbard', :].index[0]] = 0.00025        
    # alphas[inputdata.loc[inputdata["Name"] == 'Gunfleet Sand', :].index[0]] = 0.0005     
    # alphas[inputdata.loc[inputdata["Name"] == 'Gwynt y Môr', :].index[0]] = 0.001
    # alphas[inputdata.loc[inputdata["Name"] == 'Hornsea 1', :].index[0]] = 0.00025
    # alphas[inputdata.loc[inputdata["Name"] == 'Hornsea 2', :].index[0]] = 0.0002
    # alphas[inputdata.loc[inputdata["Name"] == 'Horns Rev 3', :].index[0]] = 0.0005    
    # alphas[inputdata.loc[inputdata["Name"] == 'Kaskasi', :].index[0]] = 0.0005
    # alphas[inputdata.loc[inputdata["Name"] == 'Lillgrund', :].index[0]] = 0.0005
    # alphas[inputdata.loc[inputdata["Name"] == 'Lincs', :].index[0]] = 0.001
    # alphas[inputdata.loc[inputdata["Name"] == 'Meerwind Sud/Ost', :].index[0]] = 0.0005    
    # alphas[inputdata.loc[inputdata["Name"] == 'Merkur', :].index[0]] = 0.0005 
    # alphas[inputdata.loc[inputdata["Name"] == 'Moray East', :].index[0]] = 0.0005
    # alphas[inputdata.loc[inputdata["Name"] == 'Nobelwind', :].index[0]] = 0.0005
    # alphas[inputdata.loc[inputdata["Name"] == 'Nordsee One', :].index[0]] = 0.0005                
    # alphas[inputdata.loc[inputdata["Name"] == 'Norther', :].index[0]] = 0.0005
    # alphas[inputdata.loc[inputdata["Name"] == 'Northwester 2', :].index[0]] = 0.0005    
    # alphas[inputdata.loc[inputdata["Name"] == 'Princess Amalia', :].index[0]] = 0.001  
    # alphas[inputdata.loc[inputdata["Name"] == 'Race Bank', :].index[0]] = 0.001 
    # alphas[inputdata.loc[inputdata["Name"] == 'Rampion', :].index[0]] = 0.00025 
    # alphas[inputdata.loc[inputdata["Name"] == 'Veja Mate', :].index[0]] = 0.00025
    # alphas[inputdata.loc[inputdata["Name"] == 'Walney 2', :].index[0]] = 0.00025
    # alphas[inputdata.loc[inputdata["Name"] == 'Walney Extension', :].index[0]] = 0.00025



## Original code inputs by vdLW, we made into a function, so we can call it with different alphas and different administrative boundaries

def define_alphas(inputdata):

    # A Concave hull method is used to determine the farm edge, which is not unique and depends on alpha.
    # A larger alpha means more concave. A too large alpha can result in strange results.

    alphas = [0.0001] * len(inputdata['Nt'])

    def set_alpha_if_exists(name, value):
        idx = inputdata.loc[inputdata["Name"] == name].index
        if len(idx) > 0:
            alphas[idx[0]] = value

    set_alpha_if_exists('Amrumbank West', 0.0005)
    set_alpha_if_exists('Kriegers Flak', 0.00025)
    set_alpha_if_exists('Beatrice extension', 0.0005)
    set_alpha_if_exists('Belwind Phase 1', 0.0005)
    set_alpha_if_exists('Borssele I-II', 0.0005)
    set_alpha_if_exists('DanTysk', 0.0005)
    set_alpha_if_exists('Dudgeon', 0.0005)
    set_alpha_if_exists('Galloper', 0.00015)
    set_alpha_if_exists('Gode 1 and 2', 0.0005)
    set_alpha_if_exists('Greater Gabbard', 0.00025)
    set_alpha_if_exists('Gunfleet Sand', 0.0005)
    set_alpha_if_exists('Gwynt y Môr', 0.001)
    set_alpha_if_exists('Hornsea 1', 0.00025)
    set_alpha_if_exists('Hornsea 2', 0.0002)
    set_alpha_if_exists('Horns Rev 3', 0.0005)
    set_alpha_if_exists('Kaskasi', 0.0005)
    set_alpha_if_exists('Lillgrund', 0.0005)
    set_alpha_if_exists('Lincs', 0.001)
    set_alpha_if_exists('Meerwind Sud/Ost', 0.0005)
    set_alpha_if_exists('Merkur', 0.0005)
    set_alpha_if_exists('Moray East', 0.0005)
    set_alpha_if_exists('Nobelwind', 0.0005)
    set_alpha_if_exists('Nordsee One', 0.0005)
    set_alpha_if_exists('Norther', 0.0005)
    set_alpha_if_exists('Northwester 2', 0.0005)
    set_alpha_if_exists('Princess Amalia', 0.001)
    set_alpha_if_exists('Race Bank', 0.001)
    set_alpha_if_exists('Rampion', 0.00025)
    set_alpha_if_exists('Veja Mate', 0.00025)
    set_alpha_if_exists('Walney 2', 0.00025)
    set_alpha_if_exists('Walney Extension', 0.00025)

    ###################### modification and addition in this code by SLS
    set_alpha_if_exists('Northwind', 0.0025)
    set_alpha_if_exists('Rampion', 0.00125)
    set_alpha_if_exists('Rentel', 0.001)
    set_alpha_if_exists('Robin Rigg', 0.003)

    return alphas

# Atmospheric Boundary Layer inputs defined by vdLW

Here we define the roughness lenght of the ABL and the height at which the wind rose is sampled. As defined by vdLW

In [ ]:
# Main input parameters
z0 = 10 ** (-4)  # Roughness length [m]
zRef = 100.0  # Reference height at which GWA data is taken [m]

## Wind Power Rose function
We modify the input processing function by vdLW, which was originally using the win direction frequency for the directional integration. Instead, it should be the Wind Power Rose weighted frequency, as we are dealing with the integration of power fractions.

In [ ]:
# modification of the function for assessing the wind power rose frequency distribution
# we first show the original function by vdLW, and then we show the corrected function by SLS.
# Once again, this change only impacts specific locations, where the power distribtuion as very dominant directions

# Original code by vdLW
# def calc_windresource(ds, z0, zRef, zH):
#     nWF =  len(ds.wfname)
#     ARef = np.zeros((nWF))
#     AH = np.zeros((nWF))
#     kRef = np.zeros((nWF))
#     windrose = np.zeros((nWF, 12))
#     for i in range(nWF):         
#         # Calculate wind rose frequency weighted A and k
#         ds_sub = ds.sel(wf=i, height=zRef).interp(roughness=0.0)
#         ds_sub = ds_sub.rename({'frequency': 'wdfreq'})             
#         ds_sub = ds_sub.assign_coords(west_east=ds_sub['lon'], 
#                                       south_north=ds_sub['lat'], 
#                                       sector=wk.create_sector_coords(12))
#         ds_sub = wk.spatial._point._from_scalar(ds_sub)        
#         A_combined, k_combined = wk.weibull_combined(ds_sub)
#         ARef[i] = A_combined
#         kRef[i] = k_combined              
#         windrose[i, :] = ds.sel(wf=i, height=zRef).interp(roughness=0.0)['frequency'] / ds.sel(wf=i, height=zRef).interp(roughness=0.0)['frequency'].sum()
#     # Log interpolate A at hub height
#     AH = ARef * np.log(zH / z0) / np.log(zRef / z0) 
#     return ARef, kRef, AH, windrose


# modified code by SLS
from scipy.special import gamma
def calc_windresource(ds, zRef):
    nWF =  len(ds.wfname)
    Mfactor_increase=3 # increase the number of sectors by a factor of Mfactor_increase to increase the detail of the wind rose
    windrose = np.zeros((nWF, 12*Mfactor_increase))  # 12 directions, increased by factor of Mfactor_increase to increase the detail of the wind rose
    
    for i in range(nWF):         
        ds_sub = ds.sel(wf=i, height=zRef).interp(roughness=0.0)
        ds_sub = ds_sub.rename({'frequency': 'wdfreq'})             
        ds_sub = ds_sub.assign_coords(west_east=ds_sub['lon'], 
                                      south_north=ds_sub['lat'], 
                                      sector=vdLWcode.wk.create_sector_coords(12))
        ds_sub = vdLWcode.wk.spatial._point._from_scalar(ds_sub)        
 
        # Wind Power Rose
        frequencydist = ds.sel(wf=i, height=zRef).interp(roughness=0.0)['frequency']
        lambdadist = ds.sel(wf=i, height=zRef).interp(roughness=0.0)['A']
        kdist = ds.sel(wf=i, height=zRef).interp(roughness=0.0)['k']
        Pdist = lambdadist**3 * gamma(1 + 3 / kdist) * frequencydist 
        # increase 12 sectors to 36 sectors
        Pdist36 = np.repeat(Pdist.values / Mfactor_increase, Mfactor_increase)

        # normalize
        Pdist36 = Pdist36 / np.sum(Pdist36)
        windrose[i, :] = Pdist36

    return windrose




## Define function to calculate the layout parameters
This is the function that calculates the layout parameters as defined by vdLW. The code by vdLW as some hardwired elements that decrease its flexibility. Here we copy paste the function and modify it so it can calculate a 1 to N wind farm case (the original function always calculates all wind farm cases) and checks the input data to define the for loops. Also, we removed all the plot function to keep the code short. The functionality and calculations remain the same. We do not copy the orignal code here, as it is long. The function still has the same name as the one in run.py, and the reader can check the original function in run.py.

In [ ]:
# trimmed function form teh original in run.py

def calculate_layout_metrics(Target_case, wt_xs, wt_ys, alphas, inputdata, wf_neighbors, wf_neighbors_flag, ps, windrose):
    print(' List of wind farms we will evaluate')
    for i in Target_case:
        print(' - %s : %s' % (i, inputdata['Name'][i]))

    nWF = len(inputdata['Name'])    

    Areas = []
    Nwt_rows = []
    Nwt_row_wf_neighbors = []
    inlet_all = []
    boundary_points_all = []
    normals_mean_all = []
    hulls = []
    wf_norms = []
    # Carlos modified
    Nwinddirections = len(windrose[0, :])
    print('Number of wind directions in wind rose', Nwinddirections)
    wd_windrose = np.linspace(0, 360, Nwinddirections, endpoint=False)
    # np.array([0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330])  

    print('Calculating layout metrics for %s wind farms' % nWF)

    for i in range(nWF):
        # print('Calculating layout metrics for wind farm %s' % inputdata['Name'][i])
        wt_x = np.asarray(wt_xs[i])
        wt_y = np.asarray(wt_ys[i])

        wf_norm = [wt_x.min(), wt_y.min()]
        wf_norms.append(wf_norm)
        wt_x = wt_x - wf_norm[0]
        wt_y = wt_y - wf_norm[1]      
       
        Dref = inputdata['D'][i]
        
        # Wind farm area and edge turbines
        area, boundary_points, hull = vdLWcode.calculate_wf_area_and_edge(wt_x, wt_y, ConcaveHull_alpha=alphas[i], plot=False, outfile='wflayout_polygon.pdf')
        Areas.append(area)
                
        # inlet edge turbine and turbine normals for plotting
        inlet_wd = []
        for l in range(Nwinddirections):        
            inlet, boundary_points, normals_mean = vdLWcode.calculate_front_row_wts(boundary_points, wd_windrose[l], hull)            
            inlet_wd.append(inlet)
        inlet_all.append(inlet_wd)
        boundary_points_all.append(boundary_points)
        normals_mean_all.append(normals_mean)    
        hulls.append(hull)
        
    for i in Target_case:
        print("Calculating wind farm %s" % inputdata['Name'][i])
        # Remove inlet turbines due to neighbooring wind farms with a length scale x times the turbine spacing
        # print('Removing inlet turbines due to neighboring wind farms for wind farm %s' % inputdata['Name'][i])
        S = np.sqrt(inputdata['A'][i] * 1e6) / (inputdata['D'][i] * (np.sqrt(inputdata['Nt'][i])-1))
        L = 10 * S * inputdata['D'][i]
        # print('L', L)
        wf_shadows = []
        inlet_with_wf_neighborss = []
        for l in range(Nwinddirections):
            inlet_with_wf_neighbors, wf_shadow, wf_neighbor_coords = vdLWcode.remove_inlet_turbines_wf_neighbors(i, inputdata['Name'][i], inlet_all[i][l], boundary_points_all[i], hulls, inputdata, wd_windrose[l], wf_neighbors[i], wf_neighbors_flag[i], ps, wf_norms, L)
            wf_shadows.append(wf_shadow)
            inlet_with_wf_neighborss.append(inlet_with_wf_neighbors)
   
        Nwt_row = 0
        for l in range(Nwinddirections): 
            Nwt_row = Nwt_row + len(boundary_points_all[i][:, 0][inlet_all[i][l]]) * windrose[i, l]
        Nwt_rows.append(Nwt_row)

        Nwt_row_wf_neighbor = 0        
        for l in range(Nwinddirections): 
            Nwt_row_wf_neighbor = Nwt_row_wf_neighbor + len(boundary_points_all[i][:, 0][inlet_with_wf_neighborss[l]]) * windrose[i, l]         
        Nwt_row_wf_neighbors.append(Nwt_row_wf_neighbor)         
        
           
    return Areas, Nwt_rows, Nwt_row_wf_neighbors


## Define function to overcome the wind farm labelling limitation of the code by vdLW

Most of the errors of Type 2 are associated to trating the labelling of wind turbines (belonging to a wind farm or another) as a an aerodynamic constraint, and the process where the code of vdLW forces all turbines belonging to a wind farm to be inside a single polygon.

These limitation can be overcome by some adminstrative reassigning of labels, splitting widn farms in subwindfarms, adding missing widn farms, etc. The function below does this for the specifc calculation cases. It is specific to each widn farm, but as there are few to which we are applying it, and taking into account the structure of vdLW code, it is the simplest way to do it.

In [ ]:
# these function are needed to add or subtract cases from the wind farm case list, and reassigning wind turbines to subfarms or another farm names

def duplicate_wind_farm_row(inputdata, ds, row_idx, wf_dim="wf", suffix="_2"):
    """
    Duplicate one wind farm row in both:
    - pandas DataFrame inputdata
    - xarray Dataset/DataArray ds along dimension wf_dim

    The duplicated row is inserted directly below the original.
    """

    # -----------------------------
    # 1. Duplicate pandas row
    # -----------------------------
    df = inputdata.copy().reset_index(drop=True)

    row_copy = df.iloc[[row_idx]].copy()

    # optional: change name of copied row if Name exists
    if "Name" in row_copy.columns:
        # print("Original Name:", row_copy["Name"].values[0])
        row_copy["Name"] = row_copy["Name"].astype(str) + suffix
        # print("New Name:", row_copy["Name"].values[0])

    df_new = pd.concat(
        [
            df.iloc[:row_idx + 1],
            row_copy,
            df.iloc[row_idx + 1:],
        ],
        ignore_index=True,
    )

    # -----------------------------
    # 2. Duplicate xarray row
    # -----------------------------
    ds_copy = ds.copy()

    one_wf = ds_copy.isel({wf_dim: [row_idx]}).copy()

    ds_new = xr.concat(
        [
            ds_copy.isel({wf_dim: slice(0, row_idx + 1)}),
            one_wf,
            ds_copy.isel({wf_dim: slice(row_idx + 1, None)}),
        ],
        dim=wf_dim,
    )

    # reset wf coordinate to 0, 1, 2, ...
    ds_new = ds_new.assign_coords({wf_dim: range(df_new.shape[0])})

    return df_new, ds_new



def remove_wind_farm_row(inputdata, ds, row_idx, wf_dim="wf"):
    """
    Remove one wind farm row from both:
    - pandas DataFrame inputdata
    - xarray Dataset/DataArray ds along dimension wf_dim

    The row is removed based on row_idx.
    """

    # -----------------------------
    # 1. Remove pandas row
    # -----------------------------
    df = inputdata.copy().reset_index(drop=True)

    if row_idx < 0 or row_idx >= len(df):
        raise IndexError(f"row_idx {row_idx} is out of range for inputdata")

    removed_name = df.loc[row_idx, "Name"] if "Name" in df.columns else None

    df_new = df.drop(index=row_idx).reset_index(drop=True)

    # -----------------------------
    # 2. Remove xarray row
    # -----------------------------
    if row_idx < 0 or row_idx >= ds.sizes[wf_dim]:
        raise IndexError(f"row_idx {row_idx} is out of range for ds dimension {wf_dim}")

    ds_new = xr.concat(
        [
            ds.isel({wf_dim: slice(0, row_idx)}),
            ds.isel({wf_dim: slice(row_idx + 1, None)}),
        ],
        dim=wf_dim,
    )

    # reset wf coordinate to 0, 1, 2, ...
    ds_new = ds_new.assign_coords({wf_dim: np.arange(len(df_new))})

    return df_new, ds_new, removed_name

def merge_split_farms(farm_names, *arrays, suffix="_2"):
    """
    Merge farms like 'Gemini' and 'Gemini_2'.
    The values of Gemini_2 are added to Gemini, and Gemini_2 is removed.

    farm_names: list/array of farm names
    *arrays: any number of result arrays with same length as farm_names
    """

    farm_names = list(farm_names)
    arrays = [list(a) for a in arrays]

    keep = [True] * len(farm_names)

    for i, name in enumerate(farm_names):
        if name.endswith(suffix):
            base_name = name[:-len(suffix)]

            if base_name in farm_names:
                j = farm_names.index(base_name)

                for arr in arrays:
                    arr[j] = arr[j] + arr[i]

                keep[i] = False

    new_farm_names = [name for name, k in zip(farm_names, keep) if k]
    new_arrays = [
        [value for value, k in zip(arr, keep) if k]
        for arr in arrays
    ]

    return new_farm_names, *new_arrays
# function that rearranges the input data to calculate different adminsitrative boundaries, and minimize Type 2 errors in the work of vdLW


def reassing_administrative_boundaries(inputdata, ds, wtdata, 
                                       add_Borssele_III_V=True, split_Gemini=True, split_Thorntonbank=True):
    # here we will reassign the administrative boundaries of the wind farms, to minimize Type 2 errors in the work of vdLW
    # this is a manual process, and we will do it for each wind farm in the input data

    
    # first, the cases we will always do

    # First, we will add the missing Borssele III-V, and we will call it as Borssele I-II_otherfarms, and we will assign the turbines of Borssele III-V to this new farm, 
    # the strange name is to keep using the same  duplicate_wind_farm_row function
    if add_Borssele_III_V:
        # locate the row index of Borssele I-II in the input data, to add Borsselle III-V just after
        borssele_row_idx = inputdata.index[inputdata['Name'] == 'Borssele I-II'].tolist()[0]

        # add a new row to the list of cases
        inputdata, ds =duplicate_wind_farm_row(inputdata, ds, borssele_row_idx, wf_dim="wf", suffix="_otherfarms")

        # identify the turbines of Borssele III-V in the wtdata, and reassign them to the new farm name "Borssele I-II_otherfarms"
        borssele_wtdata_idx = wtdata.index[(wtdata['wind_farm'] == 'Borssele Kavel III and IV') | (wtdata['wind_farm'] == 'Borssele Kavel V')].tolist()
        wtdata.loc[borssele_wtdata_idx, 'wind_farm'] = 'Borssele I-II_otherfarms'



    # now, we will split Gemini into two wind farms, to avoid the error of fake wind farm areas
    if split_Gemini:
        gemini_row_idx = inputdata.index[inputdata['Name'] == 'Gemini'].tolist()[0]
        inputdata, ds =duplicate_wind_farm_row(inputdata, ds, gemini_row_idx, wf_dim="wf", suffix="_2")

        # now, we will find in wtdata the rows of Gemini and longitude above 5.95, and will change the name to Gemini_2, to match the duplicated row in inputdata
        # we want to change wtdata, so maybe wiht indices, we can find the rows of Gemini and longitude above 5.95, and change the name to Gemini_2
        gemini_wtdata_idx = wtdata.index[(wtdata['wind_farm'] == 'Gemini') & (wtdata['longitude'] > 5.95)].tolist()
        wtdata.loc[gemini_wtdata_idx, 'wind_farm'] = 'Gemini_2'

    if split_Thorntonbank:
        # first, we will find in wtdata all the rows that have wind_farm that contains "Thornton Bank" and cahnge it to Thorntonbank, to match the name in inputdata
        wtdata.loc[wtdata['wind_farm'].str.contains('Thornton Bank', case=False), 'wind_farm'] = 'Thortonbank' # yes, there is a typo in the name of the farm in inputdata, but we will keep it to avoid errors in the code, and we will change the name in wtdata to match it
        thorntonbank_row_idx = inputdata.index[inputdata['Name'] == 'Thortonbank'].tolist()[0]
        inputdata, ds =duplicate_wind_farm_row(inputdata, ds, thorntonbank_row_idx, wf_dim="wf", suffix="_2")
        thorntonbank_wtdata_idx = wtdata.index[(wtdata['wind_farm'] == 'Thortonbank') & (wtdata['longitude'] > 2.95) & (wtdata['latitude'] > 51.54)].tolist()
        wtdata.loc[thorntonbank_wtdata_idx, 'wind_farm'] = 'Thortonbank_2'



    return inputdata, ds, wtdata



## Run calculations for the eleven wind farms

In [ ]:
# calculate windroses
def calculate_windrose_and_layout_parameters(inputdata, ds, wtdata, zRef):
    windrose = calc_windresource(ds, zRef)

    # calculate layout parameters
    wf_lats, wf_lons, wt_xs, wt_ys, wt_lats, wt_lons, ps = vdLWcode.get_layouts(inputdata, wtdata)  

    # calculate neighbour parameters

    wf_neighbors, wf_neighbors_flag = vdLWcode.get_wf_neighbor(wf_lats, wf_lons, inputdata['Name'], lat_dist=0.5, lon_dist=0.5)  
    return windrose, wf_lats, wf_lons, wt_xs, wt_ys, wt_lats, wt_lons, ps, wf_neighbors, wf_neighbors_flag

## Evaluation of the wind farm cases ofr which we want to have more correct values

In [ ]:
# this is the list of names that are contained in the names of the wind farm cases that we will evaluate
# it is just a bit of a manual process to select the cases we will evaluate
List_windfarm_names = ['Gemini', 'Thorton', 'Robin Rigg', 'Borssele', 'Northwind', 'Belwind', 'Northwester', 'Rentel', 'Nobelwind',
                       'Rampion', 'Norther'] # some are misplelled, but that is not important

In [ ]:
def identify_target_cases(inputdata, List_windfarm_names, print_cases=True):



    # identify in the list of cases to evaluate, therow index of cases that contains in the name the words in the List_windfarm_names, and store the row index in a list called Target_case



    Target_case = []
    for i in range(len(inputdata['Name'])):
        for name in List_windfarm_names:
            if name in inputdata['Name'][i]:
                Target_case.append(i)
                break
    # sort the Target_case list
    Target_case = sorted(Target_case)

    # print the indices and names of the cases we will evaluate
    if print_cases:
        print(' List of wind farms we will evaluate')
        for i in Target_case:
            print(' - %s : %s' % (i, inputdata['Name'][i]))

    return Target_case


In [ ]:
# 
inputdata_temp = inputdata.copy()
ds_temp = ds.copy()
wtdata_temp = wtdata.copy()

inputdata_temp, ds_temp, wtdata_temp = reassing_administrative_boundaries(inputdata_temp, ds_temp, wtdata_temp)

alphas_temp = define_alphas(inputdata_temp)


windrose, wf_lats, wf_lons, wt_xs, wt_ys, wt_lats, wt_lons, ps, wf_neighbors, wf_neighbors_flag = calculate_windrose_and_layout_parameters(inputdata_temp, ds_temp, wtdata_temp, zRef)


# display all rows and columns of the inputdata_temp dataframe, to check the changes we made in the administrative boundaries
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
# display(inputdata_temp)

Target_case = identify_target_cases(inputdata_temp, List_windfarm_names, print_cases=False)
# we now calcuate the metrics
Areas, Nwt_rows, Nwt_row_wf_neighbors = calculate_layout_metrics(Target_case, wt_xs, wt_ys, alphas_temp, inputdata_temp, wf_neighbors, wf_neighbors_flag, ps, windrose) 


#### we ned to also evaluate Thorntonbank alone, as a self neighbor

This is imposed by the structure of the code by vdLW. We need this value for the first years of oepration, which are accounted in the real capacity factor data

In [ ]:
# now, we will run a special case for Thorntonbank, 
# to determine the values for the years that there were no other neighboring farms
inputdata_thorntonbank = inputdata_temp.copy()
# eliminate all rows that do not contain "Thortonbank" in the name, to keep only the row of Thortonbank
inputdata_thorntonbank = inputdata_thorntonbank[inputdata_thorntonbank['Name'].str.contains('Thortonbank', case=False)].reset_index(drop=True)
ds_thorntonbank = ds_temp.copy().sel(wf=inputdata_thorntonbank.index)
wtdata_thorntonbank = wtdata_temp.copy()

alphas_thorntonbank = define_alphas(inputdata_thorntonbank)
windrose_thorntonbank, wf_lats_thorntonbank, wf_lons_thorntonbank, wt_xs_thorntonbank, wt_ys_thorntonbank, wt_lats_thorntonbank, wt_lons_thorntonbank, ps_thorntonbank, wf_neighbors_thorntonbank, wf_neighbors_flag_thorntonbank = calculate_windrose_and_layout_parameters(inputdata_thorntonbank, ds_thorntonbank, wtdata_thorntonbank, zRef)    
Target_case_thorntonbank = identify_target_cases(inputdata_thorntonbank, List_windfarm_names, print_cases=False)
Areas_thorntonbank, Nwt_rows_thorntonbank, Nwt_row_wf_neighbors_thorntonbank = calculate_layout_metrics(Target_case_thorntonbank, wt_xs_thorntonbank, wt_ys_thorntonbank, alphas_thorntonbank, inputdata_thorntonbank, wf_neighbors_thorntonbank, wf_neighbors_flag_thorntonbank, ps_thorntonbank, windrose_thorntonbank)   
print('Thorntonbank metrics without neighboring farms')
print('Area', Areas_thorntonbank)
print('Nwt_row', Nwt_rows_thorntonbank)
print('Nwt_row_wf_neighbors', Nwt_row_wf_neighbors_thorntonbank)





### Remove false Nobelwind overlap over Belwind and correct Nobelwind overestimation

We now correct the effect of the erroneous wind-farm region created by the vdLW code when joining the two separate parts of Nobelwind into a single region.

To isolate and quantify this error, we calculate the performance after administratively allocating the western part of Nobelwind to another wind farm. In this way, we intentionally use the vdLW coding error to our advantage.

In [ ]:
# now, we will run the special case of Belwind and Nobelwind, which are neighboring farms, 
# but are artificially merging over each other. We will untangle them in the code of vdLW
inputdata_belwind_nobelwind = inputdata_temp.copy()
# keep only the rows that contain "Belwind" or "Nobelwind" in the name, to keep only the rows of Belwind and Nobelwind
inputdata_belwind_nobelwind = inputdata_belwind_nobelwind[inputdata_belwind_nobelwind['Name'].str.contains('Belwind', case=False) | inputdata_belwind_nobelwind['Name'].str.contains('Nobelwind', case=False)].reset_index(drop=True)
ds_belwind_nobelwind = ds_temp.copy().sel(wf=inputdata_belwind_nobelwind.index)
wtdata_belwind_nobelwind = wtdata_temp.copy()

alphas_belwind_nobelwind = define_alphas(inputdata_belwind_nobelwind)


windrose_belwind_nobelwind, wf_lats_belwind_nobelwind, wf_lons_belwind_nobelwind, wt_xs_belwind_nobelwind, wt_ys_belwind_nobelwind, wt_lats_belwind_nobelwind, wt_lons_belwind_nobelwind, ps_belwind_nobelwind, wf_neighbors_belwind_nobelwind, wf_neighbors_flag_belwind_nobelwind = calculate_windrose_and_layout_parameters(inputdata_belwind_nobelwind, ds_belwind_nobelwind, wtdata_belwind_nobelwind, zRef)    
Target_case_belwind_nobelwind = identify_target_cases(inputdata_belwind_nobelwind, List_windfarm_names, print_cases=False)
Areas_belwind_nobelwind, Nwt_rows_belwind_nobelwind, Nwt_row_wf_neighbors_belwind_nobelwind = calculate_layout_metrics(Target_case_belwind_nobelwind, wt_xs_belwind_nobelwind, wt_ys_belwind_nobelwind, alphas_belwind_nobelwind, inputdata_belwind_nobelwind, wf_neighbors_belwind_nobelwind, wf_neighbors_flag_belwind_nobelwind, ps_belwind_nobelwind, windrose_belwind_nobelwind)
print('Belwind and Nobelwind metrics with neighboring farms')   
print('Area', Areas_belwind_nobelwind)
print('Nwt_row', Nwt_rows_belwind_nobelwind)
print('Nwt_row_wf_neighbors', Nwt_row_wf_neighbors_belwind_nobelwind)



# additional special case: now, the west part of Nobelwind will be alocated to another wind farm
wtdata_belwind_nobelwindeast = wtdata_temp.copy()
nobelwind_wtdata_idx = wtdata_belwind_nobelwindeast.index[(wtdata_belwind_nobelwindeast['wind_farm'] == 'Nobelwind') & (wtdata_belwind_nobelwindeast['longitude'] < 2.808) & (wtdata_belwind_nobelwindeast['latitude'] > 51.64)].tolist()
wtdata_belwind_nobelwindeast.loc[nobelwind_wtdata_idx, 'wind_farm'] = 'Northwester 2'

# run new case with the new allocation of the west part of Nobelwind to Northwester 2
winddrose_belwind_nobelwindeast, wf_lats_belwind_nobelwindeast, wf_lons_belwind_nobelwindeast, wt_xs_belwind_nobelwindeast, wt_ys_belwind_nobelwindeast, wt_lats_belwind_nobelwindeast, wt_lons_belwind_nobelwindeast, ps_belwind_nobelwindeast, wf_neighbors_belwind_nobelwindeast, wf_neighbors_flag_belwind_nobelwindeast = calculate_windrose_and_layout_parameters(inputdata_belwind_nobelwind, ds_belwind_nobelwind, wtdata_belwind_nobelwindeast, zRef)
Target_case_belwind_nobelwindeast = identify_target_cases(inputdata_belwind_nobelwind, List_windfarm_names, print_cases=False)
Areas_belwind_nobelwindeast, Nwt_rows_belwind_nobelwindeast, Nwt_row_wf_neighbors_belwind_nobelwindeast = calculate_layout_metrics(Target_case_belwind_nobelwindeast, wt_xs_belwind_nobelwindeast, wt_ys_belwind_nobelwindeast, alphas_belwind_nobelwind, inputdata_belwind_nobelwind, wf_neighbors_belwind_nobelwindeast, wf_neighbors_flag_belwind_nobelwindeast, ps_belwind_nobelwindeast, winddrose_belwind_nobelwindeast)
print('Belwind and Nobelwind metrics with the west part of Nobelwind allocated to Northwester 2')
print('Area', Areas_belwind_nobelwindeast)
print('Nwt_row', Nwt_rows_belwind_nobelwindeast)
print('Nwt_row_wf_neighbors', Nwt_row_wf_neighbors_belwind_nobelwindeast)

# third special case, with all neighbours, but once again we will allocate the west part of Nobelwind to Northwester 2, to see the impact of this change in the metrics
inputdata_belwind_nobelwindeast_neighbors = inputdata_temp.copy()
ds_belwind_nobelwindeast_neighbors = ds_temp.copy()
wtdata_belwind_nobelwindeast_neighbors = wtdata_belwind_nobelwindeast.copy()
alphas_belwind_nobelwindeast_neighbors = define_alphas(inputdata_belwind_nobelwindeast_neighbors)
windrose_belwind_nobelwindeast_neighbors, wf_lats_belwind_nobelwindeast_neighbors, wf_lons_belwind_nobelwindeast_neighbors, wt_xs_belwind_nobelwindeast_neighbors, wt_ys_belwind_nobelwindeast_neighbors, wt_lats_belwind_nobelwindeast_neighbors, wt_lons_belwind_nobelwindeast_neighbors, ps_belwind_nobelwindeast_neighbors, wf_neighbors_belwind_nobelwindeast_neighbors, wf_neighbors_flag_belwind_nobelwindeast_neighbors = calculate_windrose_and_layout_parameters(inputdata_belwind_nobelwindeast_neighbors, ds_belwind_nobelwindeast_neighbors, wtdata_belwind_nobelwindeast_neighbors, zRef)
Target_case_belwind_nobelwindeast_neighbors = identify_target_cases(inputdata_belwind_nobelwindeast_neighbors, ['Belwind', 'Nobelwind'], print_cases=False)
Areas_belwind_nobelwindeast_neighbors, Nwt_rows_belwind_nobelwindeast_neighbors, Nwt_row_wf_neighbors_belwind_nobelwindeast_neighbors = calculate_layout_metrics(Target_case_belwind_nobelwindeast_neighbors, wt_xs_belwind_nobelwindeast_neighbors, wt_ys_belwind_nobelwindeast_neighbors, alphas_belwind_nobelwindeast_neighbors, inputdata_belwind_nobelwindeast_neighbors, wf_neighbors_belwind_nobelwindeast_neighbors, wf_neighbors_flag_belwind_nobelwindeast_neighbors, ps_belwind_nobelwindeast_neighbors, windrose_belwind_nobelwindeast_neighbors)
print('Belwind and Nobelwind metrics with the west part of Nobelwind allocated to Northwester 2 and with all neighbors')
print('Area', Areas_belwind_nobelwindeast_neighbors)
print('Nwt_row', Nwt_rows_belwind_nobelwindeast_neighbors)
print('Nwt_row_wf_neighbors', Nwt_row_wf_neighbors_belwind_nobelwindeast_neighbors)



## Result bookkeeping

At this stage, we merge the administratively split wind farms back into their original combined entities (e.g., `Gemini` and `Gemini_2`) by summing the corresponding results and removing the duplicate bookkeeping entries.

We also merge the special cases generated through administrative name manipulation, where we intentionally exploited the limitations of the vdLW code to isolate and mitigate specific type-2 errors.

In [ ]:
# make a table with the results, including the names of the wind farms
results_df = pd.DataFrame({
    'Name': [inputdata_temp['Name'][i] for i in Target_case],
    'Area_km2': [Areas[i] for i in Target_case],
    'Nwt_rows': Nwt_rows,
    'Nwt_row_wf_neighbors': Nwt_row_wf_neighbors
})

# for each widn farm, based on the name, count the number of widn trubines in wtdata, and add it as a column in results_df
results_df['Nwt'] = results_df['Name'].apply(lambda x: len(wtdata_temp[wtdata_temp['wind_farm'] == x]))


display(results_df)

In [ ]:
results_merge_corrected_df = results_df.copy()

# merge Gemini and Gemini_2 in the results_merge_corrected_df, by adding the values of Gemini_2 to Gemini, and removing the row of Gemini_2
results_merge_corrected_df = results_merge_corrected_df.groupby(results_merge_corrected_df['Name'].str.replace('_2', ''), as_index=False).agg({
    'Area_km2': 'sum',
    'Nwt_rows': 'sum',
    'Nwt_row_wf_neighbors': 'sum',
    'Nwt': 'sum'
})
# for mergining Gemini, the value of Nwt_rows is equal to the value of Nwt_row_wf_neighbors, there are no other widn farms nearby
results_merge_corrected_df.loc[results_merge_corrected_df['Name'] == 'Gemini', 'Nwt_rows'] = results_merge_corrected_df.loc[results_merge_corrected_df['Name'] == 'Gemini', 'Nwt_row_wf_neighbors']

# for Thorntonbank, the value of Nwt_rows is equal to thesum of Nwt_row_wf_neighbors_thorntonbank
results_merge_corrected_df.loc[results_merge_corrected_df['Name'] == 'Thortonbank', 'Nwt_rows'] = np.sum(Nwt_row_wf_neighbors_thorntonbank)

# Correct Belwind for the case with neighbours
results_merge_corrected_df.loc[results_merge_corrected_df['Name'] == 'Belwind Phase 1', 'Nwt_row_wf_neighbors'] = Nwt_row_wf_neighbors_belwind_nobelwindeast_neighbors[1]


# For Nobelwind, we need to decrease the value of Nwt, to compensate that vdLW apply a factor of 2.5 to all edge turbines. 
# However, the west part of Noberwind is a single row of turbines, so we will decrease the value of Nwt by the number of turbines
#  in the west part of Nobelwind, to avoid that, locally, Nfree is larget than the actual number of turbines in the farm, which would be a mistake.
# first, the case without neighbous that is the case with Belwidn as a single neighbor
Nwt_Nobelwind_east = Nwt_row_wf_neighbors_belwind_nobelwindeast[0] 
Nwt_Nobelwind_west = results_merge_corrected_df.loc[results_merge_corrected_df['Name'] == 'Nobelwind', 'Nwt_rows'].values[0] - Nwt_Nobelwind_east
Number_turbines_Nobelwind_west = len(wtdata_temp[(wtdata_temp['wind_farm'] == 'Nobelwind') & (wtdata_temp['longitude'] < 2.808) & (wtdata_temp['latitude'] > 51.64)])
# print('Number_turbines_Nobelwind_west', Number_turbines_Nobelwind_west)
# print('Nwt_Nobelwind_west', Nwt_Nobelwind_west)
# print('Nwt_Nobelwind_east', Nwt_Nobelwind_east)
# because Number_turbines_Nobelwind_west<2.5*Nwt_Nobelwind_west, then Nwt_Nobelwind_west = Number_turbines_Nobelwind_west/2.5, to avoid that Nfree is larger than the actual number of turbines in the farm
if Number_turbines_Nobelwind_west < 2.5 * Nwt_Nobelwind_west:
    Nwt_Nobelwind_west = Number_turbines_Nobelwind_west / 2.5

results_merge_corrected_df.loc[results_merge_corrected_df['Name'] == 'Nobelwind', 'Nwt_rows'] = Nwt_Nobelwind_east + Nwt_Nobelwind_west

# and now, the same for the case with all neighbors
Nwt_Nobelwind_east_neighbors = Nwt_row_wf_neighbors_belwind_nobelwindeast_neighbors[0]
Nwt_Nobelwind_west_neighbors = results_merge_corrected_df.loc[results_merge_corrected_df['Name'] == 'Nobelwind', 'Nwt_row_wf_neighbors'].values[0] - Nwt_Nobelwind_east_neighbors
# print('Nwt_Nobelwind_west_neighbors', Nwt_Nobelwind_west_neighbors)
# print('Nwt_Nobelwind_east_neighbors', Nwt_Nobelwind_east_neighbors)
if Number_turbines_Nobelwind_west < 2.5 * Nwt_Nobelwind_west_neighbors:
    Nwt_Nobelwind_west_neighbors = Number_turbines_Nobelwind_west / 2.5
results_merge_corrected_df.loc[results_merge_corrected_df['Name'] == 'Nobelwind', 'Nwt_row_wf_neighbors'] = Nwt_Nobelwind_east_neighbors + Nwt_Nobelwind_west_neighbors

# elimate the row of Borssele I-II_otherfarms, that we created just to allocate the turbines of Borssele III-V, because it is not a real farm, and we will not use it in the analysis
results_merge_corrected_df = results_merge_corrected_df[results_merge_corrected_df['Name'] != 'Borssele I-II_otherfarms'].reset_index(drop=True)

display(results_merge_corrected_df)

## Uploading the original edge Nfree factors by vdLW

To recall: we are only going to correct some of the Type 2 errors in some of the cases, as we are limited by nto changing the core code and method by vdLW.

Therefore, we will upload the results by vdLW for the Nfree edge factors


## Read Nedge values from Output/Input files by vdLW

vdLW, with their code, create a file named "Output.csv", which contains both inputs and outputs used by vdLW, including the inputs used by SLS.

An important note is that vdLW do not use all the wind farm cases. vdLW choose not to use the cases of the wind farms "Baltic 1", "Baltic 2", "Princess Amalia", "Luchterduinen". Therefore, after reading the files, we will need to remove those cases.

In [ ]:
# read output/input data from vdLW's code
df = pd.read_csv("output.csv")
df_results_vdLW = df.copy() # this is just not to destroy the original dataframe, as we will be modifying it in the next steps

# we will eliminate the cases of Baltic I and II and Princess Amalia and Lucherduinen, as they are not evaluated by vdLW's work
df_results_vdLW = df_results_vdLW[~df_results_vdLW["Name"].isin(["Baltic 1", "Baltic 2", "Princess Amalia", "Luchterduinen"])]

# display the entire database for the first evaluation, indicating explicitly to plot all the rows, to avoid the Jupyter notebook from truncating the output
# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
# display(db_first_eval)

In [ ]:
# we read the number wind turbines in the wind farm
Windfarm_N_turbines = df_results_vdLW["Nt"].values

# we again read the values of the correction factor used by SLS 2026 to represent the fraction of power equivalent to isolated turbine afactor_SLS2026
NFreeSLStemp = np.array(df_results_vdLW['Nrowscale parameter'] * df_results_vdLW['Nturb_frontal_area'], dtype=float)
afactor_SLS2026 = NFreeSLStemp / (Windfarm_N_turbines)   


# we now read the estimates by vdLW for the edge turbines, with and without neighbors
Nedge_neighbors_vdLW = df_results_vdLW["Nwt_row_wf_neighbors"].values
Nedge_without_neighbours_vdLW = df_results_vdLW["Nwt_rows"].values

# vdLW use a 2.5 multiplicative factor for the edge turbines, and use only the case with neighbours.
m_afactor_vdLW = 2.5

# we will also calculate the afactor by vdLW. We do not need for creating figures 4 and 5, 
# but we will do it for connection to figure 3, which although published in the other notebook, we will reproduce here

# vdLW use a 2.5 multiplicative factor for the edge turbines, and use only the case with neighbours.
m_afactor_vdLW = 2.5
a_factor_vdLW_Nfree=m_afactor_vdLW*Nedge_neighbors_vdLW/(Windfarm_N_turbines)



## Correct Nedge with values calculated above

In [ ]:
# first, we create arrays for the Nedge metrics, and equal them to the values of vdLW

Nedge_neighbors_vdLW_corrected = Nedge_neighbors_vdLW .copy()
Nedge_without_neighbours_vdLW_corrected = Nedge_without_neighbours_vdLW.copy()

# now, we will correct the values of Nedge_neighbors_vdLW_corrected and Nedge_without_neighbours_vdLW_corrected for the 
# cases of Gemini, Thortonbank, Belwind and Nobelwind, etc, based on the calculations above

for i in range(len(results_merge_corrected_df)):
    name = results_merge_corrected_df.iloc[i]["Name"]
    # print(f"Checking wind farm {name} for correction of Nedge due to geometry")

    matches = np.where(df_results_vdLW["Name"].to_numpy() == name)[0]

    if len(matches) > 0:
        pos = matches[0]   # row position, safe for iloc and arrays

        # print(f"Position of wind farm {name} in df_results_vdLW: {pos}")
        # print(
        #     f"Nwt_rows and Nwt_row_wf_neighbors before correction for wind farm {name}: "
        #     f"{df_results_vdLW.iloc[pos]['Nwt_rows']}, "
        #     f"{df_results_vdLW.iloc[pos]['Nwt_row_wf_neighbors']}"
        # )

        # print(
        #     f"Original Nedge_isolated: {Nedge_isolated[pos]}, "
        #     f"Original Nedge_Neighbours: {Nedge_Neighbours[pos]}"
        # )

        Nedge_without_neighbours_vdLW_corrected [pos] = results_merge_corrected_df.iloc[i]["Nwt_rows"]
        Nedge_neighbors_vdLW_corrected[pos] = results_merge_corrected_df.iloc[i]["Nwt_row_wf_neighbors"]
        # print(
        #     f"Corrected Nedge_isolated: {Nedge_without_neighbours_vdLW_corrected[pos]}, "
        #     f"Corrected Nedge_Neighbours: {Nedge_neighbors_vdLW_corrected[pos]}"
        # )

    else:
        print(f"Wind farm {name} not found in results_merge_corrected_df, skipping correction")



## Correct the multiply term of edge turbines, to keep it consistent with being below the actual number of turbines in the farm

This is the correction for Error Type 2 - Error 11 — Assuming Unrealistically Large Numbers of Turbines Exposed to Clean Inflow

The vdLW methodology estimates the number of turbines exposed to clean inflow using the relation: Nfree = 2.5M, where M is the number of detected inlet-edge turbines. This factor of 2.5 is not a general physical constant. It comes from applying the finite-size correction with a = 5, corresponding to the characteristic finite-farm correction parameter used in SLS for a square wind-farm scaling approximation, and
assuming that the exposed edge of a square wind farm corresponds to approximately 2√N turbines. This leads to the approximation that the number of turbines exposed to clean inflow is about 2.5 times the number of detected inlet-edge turbines. However, applying this relation as a fixed multiplier to arbitrary wind-farm geometries is a misinterpretation of both SLS and the earlier work by Sørensen and Larsen.
The correction must depend on the total number.

Here, for wind farms smaller than 25 turbines, we correct the value.

In [ ]:
# define a simple approximation for very small values of number of wind turbines in the wind farm

def simple_afactor_model(Nt):
    N_squared_root = np.sqrt(Nt)
    # simple formula based on the results of Larsen and Sorensen, with a correction factor to ensure 
    # that at N_squared_root=1, the fraction of clean flow is 1.0. This approximation loses validity for any large wind farm
    a = 0.5788*(N_squared_root-1)+1.0

    # we cap a at 6 based on what was our knowledge then of the slope
    a = np.minimum(a, 5)/2

    if Nt >25:
        a =np.nan  # for large wind farms, this simple model is not valid, so we set it to NaN

    return a

In [ ]:
## Correct the multiply term of edge turbines, to keep it consistent with being below the actual number of turbines in the farm

m_a_factor_vdLW_Nfree_corrected = np.zeros_like(Nedge_neighbors_vdLW_corrected)+2.5

for i in range(len(Nedge_neighbors_vdLW_corrected)):
    Nt = Windfarm_N_turbines[i]
    if Nt < 25:
        print(f"Correcting multiplicative factor for edge turbines for wind farm {df_results_vdLW.iloc[i]['Name']} with Nt={Nt}")
        # for small wind farms, we will use the simple approximation for the fraction of clean flow, to correct the multiplicative factor of edge turbines
        m_a_factor_vdLW_Nfree_corrected[i] = simple_afactor_model(Nt)



## Correct Type 2 Error 10— Ignoring the Temporal Evolution of Wind-Farm Clusters

The vdLW evaluation assumes a fixed 2025 wind-farm configuration, despite the measured production data spanning years
during which neighboring wind farms were progressively constructed and commissioned. The vdLW simulations therefore
compare measured production from partially developed wind-farm clusters against aerodynamic environments corresponding
to much later cluster-development stages.
As a consequence, the aerodynamic environment used in the vdLWsimulations does not correspond to the real aerodynamic
environment experienced by the wind farms during the years covered by the measured production data.
This creates a mismatch between:
– the real neighboring wind farms present during operation,
– and the neighboring wind farms included in the vdLW simulations.
The error is particularly important for rapidly evolving offshore wind-farm clusters, where neighboring wind farms were
added progressively over time and the aerodynamic environment evolved continuously during the measurement period.

In this part, we upload a table with an indication of the number of years the wind farm was without and with neighbours, based on the real capacity factor database used.

In [ ]:
# load table with the breakdown of year of isolated wind farma and with neighbours
infile_years_neighbours='year_correction_isolated_neighbours.xlsx'
df_years_neighbours = pd.read_excel(infile_years_neighbours, sheet_name='Sheet1')

years_isolated = df_years_neighbours['Years isolated'].values
years_neighbours = df_years_neighbours['Years neighbours'].values

Nedge_corrected = Nedge_neighbors_vdLW_corrected * years_neighbours / (years_isolated + years_neighbours) + Nedge_without_neighbours_vdLW_corrected * years_isolated / (years_isolated + years_neighbours)

## Calculate correct capacity factors

We have now setup all the corrections. We can now calculate the correct capacity factor. We will upload the corrected capacity factors for the isolated turbine and infinite wind farm for each wind farm case, without Type 1 errors, calculated in the other notebook

In [ ]:
# load values free of Type 1 errors
infile_clean_of_type1_errors = 'vdLW_corrected_capacity_factors.xlsx'
df_clean_of_type1_errors = pd.read_excel(infile_clean_of_type1_errors, sheet_name='Sheet1')

# display all columns and rows of the dataframe, to check the values
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
# display(df_clean_of_type1_errors)

CFisolated_vdLW_free_of_type1_errors = np.array(df_clean_of_type1_errors['Cf_Free_vdLW_correct_all'].values, dtype=float)
CFinfinitewf_vdLW_free_of_type1_errors = np.array(df_clean_of_type1_errors['Cf_Inf_vdLW_correct_all'].values, dtype=float)
CFmodel_vdLW_free_of_type1_errors = np.array(df_clean_of_type1_errors['Cf_Model_vdLW_correct_all'].values, dtype=float)

# we also upload the cased for Uin=0 and Uout=infinity, to later determine the Wind Farm Wind Factor

CFisolated_Uin0_Uoutinf = np.array(df_clean_of_type1_errors['Cf_Free_vdLW_correct_Uin0_Uoutinf'].values, dtype=float)
CFinfinitewf_Uin0_Uoutinf = np.array(df_clean_of_type1_errors['Cf_Inf_vdLW_correct_Uin0_Uoutinf'].values, dtype=float)



In [ ]:
# calculate the Capacity factor corrected of Type 1 errors and with some Type 2 errors mitigated

fraction_Nfree =  m_a_factor_vdLW_Nfree_corrected * Nedge_corrected / Windfarm_N_turbines

CF_vdLW_free_Type1_and_2_errors = CFisolated_vdLW_free_of_type1_errors * fraction_Nfree + CFinfinitewf_vdLW_free_of_type1_errors * (1-fraction_Nfree)




## Read the results published by SLS 2026, from comparison and generation of the figures

We already removed the wind farms "Baltic 1", "Baltic 2", "Princess Amalia", "Luchterduinen" because vdLW removed them from their work

In [ ]:
# read from excel file the dataframe of SLS_2026
df_SLS_2026 = pd.read_excel("SLS_2026_without_Baltic1_2_LuchD_PAM.xlsx", sheet_name="Sheet1")

# createa array of capacity factors from the dataframe of SLS_2026, 
# this will be used to compare with the results of the corrected vdLW model
SLS_2026_CF = np.array(df_SLS_2026["Capacity Factor model"].values, dtype=float)/100

# also read the real values of capacity factor

SLS_2026_CF_real = np.array(df_SLS_2026["Capacity Factor real"].values, dtype=float)/100


## Create Figure 5



In [ ]:
# simple code for the linear fit

def linear_fit(x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    # fit y = a*x + b
    a, b = np.polyfit(x, y, 1)

    # predictions
    y_pred = a * x + b

    # R^2
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r2 = 1 - ss_res / ss_tot

    return a, b, r2

### Now, we plot Figure 5

Note that the regression shown here is slightly better than the one published in the rebuttal document. While cleaning and restructuring the code, we identified a small error in one data point related to the merging of a split wind farm. After correcting this bookkeeping issue, the resulting regression became marginally stronger.

In [ ]:
# determine terms of linear regression between the capacity factor of vdLW with Type 1 error corrected Nfree and the SLS_2026_CF, 
# to see how well they correlate, and to see if there is a bias in the results
a_reg,b_reg,r2_reg = linear_fit(SLS_2026_CF, CF_vdLW_free_Type1_and_2_errors)
print("Linear fit: a =", a_reg, "b =", b_reg, "R^2 =", r2_reg)

# Plot figure 5

plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams.update({'font.size': 14}) # fontsize 14 for all labels and ticks


# we will now plot in the comparison between model and real CF
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(SLS_2026_CF*100,CF_vdLW_free_Type1_and_2_errors*100, 'o',color='orange',  
        label=f'vdLW Nfree methodology, Type 1 errors corrected, some Type 2 errors corrected for some cases.\n Linear regression fit: a={a_reg:.2f}, b={b_reg:.2f}, R^2={r2_reg:.2f}')
ax.plot([0., 100.], [0., 100.], "k--", label="1:1 line")

ax.set_xlim(20, 70)
ax.set_ylim(20, 70)
ax.set_xlabel('Capacity Factor SLS (%)')
ax.set_ylabel('Capacity Factor vdLW (%)')
ax.legend()
# ax.set_title('Comparison of modelled and 2025 and Paul')
ax.grid()
plt.show()

## Plot Figure 6

Here we plot Figure 6, the corrected results using the vdLW framework, plotted against the theoretical limit.

First, we calculate the theoretical limit curves, also using the corrected vdLW code, then, the equivalent Wind Farm Wind Factor, 
and then the full plot

In [ ]:
# again, we copy and correct the function of vdLW to eliminate the eps2 error

from scipy.special import gamma, gammainc 
from scipy.optimize import fsolve

class MinimalisticPredictionModel():
    """Sørensen, J.N.; Larsen, G.C.
    A Minimalistic Prediction Model to Determine Energy Production and Costs of Offshore Wind Farms.
    Energies 2021, 14, 448. https://doi.org/10.3390/en14020448"""

    def __init__(self, correction_factor, latitude, CP, Uin, Uout, rho, correct_eps2=False):
        ########################### Start of note by SLS ###########################
        # The original code by vdLW had the function header as: def __init__(self, correction_factor, latitude, CP, Uin, Uout, rho):
        # 
        # We have added an optional argument 'correct_eps2' to the function header. This argument is a boolean.
        # If False (default), the function will use the original calculation for eps2 as given by vdLW.
        # If True, the function will use the correct formula for eps2
        #
        ########################### End of note by SLS   ###########################
        """
        Parameters
        ----------
        correction_factor : int, float or function
            Finite-size wind farm corrrection which multiplied with sqrt(Nturb) gives
            the number of wind turbines exposed to the free wind
        latitude : int or float
            latitude [deg] used to calculate the coriolis parameter
        CP : float, optional
            Wind turbine power coefficient
        Uin : int or float, optional
            Wind turbine cut-in wind speed
        Uout : int or float, optional
            Wind turbine cut-out wind speed
        rho : float, optional
            Air density
        """

        self.CP = CP
        self.Uin = Uin
        self.Uout = Uout
        omega = 2 * np.pi / (24 * 60 * 60)  # earth rotation speed
        self.f = 2 * omega * np.sin(np.deg2rad(latitude))
        self.correction_factor = correction_factor
        self.rho = rho

        ################ Start of code alteration by SLS to add correct_eps2 argument, see note at the beginning of the class #################
        self.correct_eps2 = correct_eps2
        ################ End of code alteration by SLS to add correct_eps2 argument #################
        
    def predict(self, Pg, CT, D, H, z0, Aw, kw, Nturb, Area):
        """
        Inputs:
            Pg    - [W] Nameplate capacity (generator power)
            CT    - [-] Thrust coefficient
            D     - [m] Rotor diameter
            H     - [m] Tower height
            z0    - [m] roughness length
            Aw    - [m/s] Weibull scale parameter
            kw    - [-] Weibull shape parameter
            Nturb - [-] Number of turbines
            Area  - [m2] Area of wind farm

        Outputs:
            power - [Wh] Annual energy production of the wind farm
            ws_eff - [m/s] Effective mean wind speed including wakes
        """

        kappa = 0.4  # [-] Von Karman constant
        Uin, Uout = self.Uin, self.Uout

        # factor defined by Frandsen, should be used instead of f in eq 13 and 19 (typos in paper)
        fm = self.f * np.exp(4)
        delta = np.log(H / z0)  # eq 19

        # Mean spacing between wt in diameters, eq 8
        S = np.sqrt(Area) / (D * (np.sqrt(Nturb) - 1))

        # Rated wind speed [m/s], eq 4
        Ur = (8 * Pg / (self.rho * np.pi * D**2 * self.CP))**(1 / 3)

        # Power modeled as P = alpha * U^3 + beta, eq 1
        alpha = Pg / (Ur**3 - Uin**3)  # [(m/s)^-3] eq 2
        beta = -Pg * Uin**3 / (Ur**3 - Uin**3)  # [-], eq 2

        Uh0 = Aw * gamma(1 + 1 / kw)  # [m/s] Mean velocity at hub height
        Ctau = np.pi * CT / (8 * S * S)  # [-] Wake parameter, rotor ct smeared on WT area
        nu = np.sqrt(0.5 * Ctau) * D / (kappa**2 * H) * delta  # [-] wake eddy viscosity

        # Finite-size wind farm corrrection, section 2.5
        correction_factor = self.correction_factor
        if hasattr(correction_factor, '__call__'):
            correction_factor = correction_factor(Uh0, S, Nturb)
        Nfree = correction_factor * np.sqrt(Nturb)  # Number of wt exposed to the free wind
        Nfree = np.minimum(Nfree, Nturb)  # To make sure Nfree/Nturb is not larger than one, not yet implemented in PyWake

        # Geostrophic wind speed
        G_last = Uh0
        for n in range(10):
            G = Uh0 * (1 + np.log(G_last / (fm * H)) / delta)
            dG = abs(G - G_last)
            if dG < 1e-5:
                break
            G_last = G

        gam = np.log(G / (fm * H))  # eq 19

        # Mean velocity at hub height without wake effects from geostrophic wind
        Uh0 = G / (1 + gam / kappa * np.sqrt((kappa / delta)**2))  # eq 13, ct=0

        # Power without wake effects, eq 16 modified by
        # - add gamma(1 + 3 / kw) to cancel out normalization in scipy's gammainc
        # - gammainc terms swapped (typo in paper) 
        def get_Py(Aw, Aw_out):  # Yearly power
            return alpha * Aw**3 * gamma(1 + 3 / kw) * (gammainc(1 + 3 / kw, (Ur / Aw)**kw) - gammainc(1 + 3 / kw, (Uin / Aw)**kw)) +\
                beta * (np.exp(-(Uin / Aw)**kw) - np.exp(-(Ur / Aw)**kw)) + \
                Pg * (np.exp(-(Ur / Aw)**kw) - np.exp(-(Uout / Aw_out)**kw))

        P_y = get_Py(Aw, Aw)
 
        # Without cutin and cutout
        # def get_Py(Aw):  # Yearly power
        #     x = (Ur / Aw) ** kw
        #     ks = 1 + 3 / kw
        #     return Pg * (x ** (1.0 - ks) * gamma(ks) * gammainc(ks, x) + np.exp(-x))
        # def get_Py(Aw):  # Yearly power
        #     x = (Ur / Aw)
        #     ks = 1 + 3 / kw
        #     return Pg * (x ** (-3) * gamma(ks) * gammainc(ks, x ** kw) + np.exp(-x ** kw))                      
        # P_y = get_Py(Aw)

        # Mean velocity at hub height with wake effects
        z0_lo = z0  # / (1 - D / (2 * H))**(nu / (1 + nu))  # ???
        Uh = G / (1 + gam * np.sqrt(Ctau + (kappa / np.log(H / z0_lo))**2) / kappa)

        # eq 18. The paper states 3/2 instead of 3.2 which is either a typo or an initial guess
        # eps2 corresponds to eps(Uout) in paper and eps2(Ur)=eps1
        eps1 = (1 + gam / delta) / (1 + gam / kappa * np.sqrt(Ctau + (kappa / delta)**2))
        eps2 = (1 + gam / delta) / (1 + gam / kappa * np.sqrt(Ctau * (Ur / Uh)**3.2 + (kappa / delta)**2))


        ################# Start of code alteration by SLS to correct eps2, see note at the beginning of the class #################
        if self.correct_eps2: # 
            # use the correct formulation
            eps2 = (1 + gam / delta) / (1 + gam / kappa * np.sqrt(Ctau * (Ur / Uout)**3.2 + (kappa / delta)**2))
        ################### End of code alteration by SLS to correct eps2 #################




        # print('e', eps1, correction_factor)
        # Power production with wake effects
        P_WFy = get_Py(eps1 * Aw, eps2 * Aw)

        power = ((Nturb - Nfree) * P_WFy + Nfree * P_y)
        ws_eff = ((Nturb - Nfree) * Uh + Nfree * Uh0) / Nturb
        
        # Phi without cutin and cutout 
        def get_phi(x):  # Yearly power
            # x = U_r / (Uh0 eps)
            ks = 1 + 3 / kw      
            Gm = gamma(1 + 1 / kw)
            return Pg * ((x * Gm) ** (-3) * gamma(ks) * gammainc(ks, (x * Gm) ** kw) + np.exp(-(x * Gm) ** kw)) - power / Nturb
            
        root = fsolve(get_phi, x0=1)  # initial guess        
        phi = root[0]

        return power, ws_eff, P_y, P_WFy, Uh, G, phi, eps1, Ur # small modification to return eps1 and Ur 



In [ ]:
# we create an instance of the class, with Uin=0 and Uout=infinity, and eps2 corrected

# parameters as defined by vdLW, these are the same for all cases, as they are not case specific, 
# but they are used as input for the predict function, so we define them here
vdLW_z0 = 10 ** (-4)  # Roughness length [m]
vdLW_zRef = 100.0  # Reference height at which GWA data is taken [m]
vdLW_kw = 2.4  # Weibull shape parameter [-]
vdLW_rho = 1.225  # Air density [kg/m^3]
vdLW_lat = 55  # Latitude [degree]
vdLW_CP = 0.46  # Turbine power coefficient [-]
vdLW_CT = 0.75  # Turbine thrust coefficient [-]
vdLW_Uin = 0.0  # Cut-in wind speed [m/s]
vdLW_Uout = 1e6  # Cut-out wind speed [m/s]

vdLW_model = MinimalisticPredictionModel(correction_factor=0, latitude=vdLW_lat, CP=vdLW_CP, Uin=0, Uout=np.inf, rho=vdLW_rho, correct_eps2=True)


# now, we setup the limit curve


Turbine_rated_power_lim = 15e6
CT_lim = vdLW_CT
Turbine_rotor_diameter_lim = 240
Turbine_hub_height_lim = 150
z0_lim = vdLW_z0
Wind_lambda_lim = 15
kw_lim = vdLW_kw
Windfarm_area_lim = (Turbine_rotor_diameter_lim*5*100)**2
Windfarm_N_turbines_lim = np.linspace(10, 1000, 100)**2

power_lim = np.zeros(len(Windfarm_N_turbines_lim))
ws_eff_lim = np.zeros(len(Windfarm_N_turbines_lim))
P_y_lim = np.zeros(len(Windfarm_N_turbines_lim))
P_WFy_lim = np.zeros(len(Windfarm_N_turbines_lim))
Uh_lim = np.zeros(len(Windfarm_N_turbines_lim))
G_lim = np.zeros(len(Windfarm_N_turbines_lim))
phi_lim = np.zeros(len(Windfarm_N_turbines_lim))
eps1_lim = np.zeros(len(Windfarm_N_turbines_lim))
Ur_lim = np.zeros(len(Windfarm_N_turbines_lim))

for i in range(len(Windfarm_N_turbines_lim)):
    power_lim[i], ws_eff_lim[i], P_y_lim[i], P_WFy_lim[i], Uh_lim[i], G_lim[i], phi_lim[i], eps1_lim[i], Ur_lim[i] = vdLW_model.predict(
        Turbine_rated_power_lim, CT_lim, Turbine_rotor_diameter_lim, Turbine_hub_height_lim, z0_lim, Wind_lambda_lim, kw_lim, 
        Windfarm_N_turbines_lim[i], Windfarm_area_lim)


CF_lim =  power_lim / (Turbine_rated_power_lim * Windfarm_N_turbines_lim)
Uinf_lim = Wind_lambda_lim * gamma(1 + 1/kw_lim) 
WFWF_lim = Ur_lim / Uh_lim


# create interpolation function for the upper limit curve, to be able to calculate the phi values for the cases in the database
CF_function_of_WFWF = interp1d(WFWF_lim, CF_lim, bounds_error=False, fill_value="extrapolate")
WFWF_function_of_CF = interp1d(CF_lim, WFWF_lim, bounds_error=False, fill_value="extrapolate")



### Calculate the value of the Wind Farm Wind Factor $\phi$

In [ ]:
# first, we calculate the capacity factor value based in operation Uin=0 to Uinf=infinity, to be able to determine the Wind Farm Wind Factor

CF_vdLW_free_Type1_and_2_errors_Uin0_Uoutinfinity = CFisolated_Uin0_Uoutinf * fraction_Nfree + CFinfinitewf_Uin0_Uoutinf * (1-fraction_Nfree)

# determine the Wind Farm Wind Factor for each case in the database

WFWF_vdLW_free_Type1_and_2_errors = WFWF_function_of_CF(CF_vdLW_free_Type1_and_2_errors_Uin0_Uoutinfinity)





### Plot Figure 6

In [ ]:
## plot CF_lim vs WFWF_lim
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(WFWF_lim, CF_lim*100, '-', color='navy', label=r'Theoretical limit curve')
ax.plot(WFWF_lim, CF_lim*100 * 0.9, '--', color='blue', label=r'90% Theoretical limit curve')
# ax.scatter(WFWF_random, CF_random*100, color='cyan', label=r'Random points on upper limit curve')
# ax.plot(wf_previous, Cf_Model_paul_correct_lambda_eps_Uin_Uout_lat*90, 'o', color='orange', label=r'previous wfwf')
# ax.plot(wf_corrected, Cf_Model_paul_correct_lambda_eps_Uin0_Uoutinf_lat*90, 'o', color='red', label=r'corrected wfwf')
ax.plot(WFWF_vdLW_free_Type1_and_2_errors, SLS_2026_CF_real*100, '*', color='green',alpha=0.95, label=r'real data, vdLW methodology with corrections')
# ax.plot(wf_corrected, CF_2025_real*100, '+', color='red', label=r'corrected wfwf')
ax.set_xlim(1, 2)
ax.set_ylim(20, 60)
ax.set_xlabel('Wind Farm Wind Factor ')
ax.set_ylabel('Capacity Factor (CF) %')
ax.legend()
ax.grid(True)
plt.show()